# AHC Visual Intelligence Hackathon — detection pipeline

Real-time video anomaly detection over drone / CCTV / dashcam footage, in one
notebook that runs entirely on Kaggle.

**Session options:** Accelerator **GPU T4 ×2**, Internet **On**.
**Add data:** the AHC train+test pack, attached as a Kaggle Dataset.

## The architecture

A three-tier cascade, following *Cerberus* (arXiv 2510.16290), which the problem
statement blesses directly ("a lightweight always-on stage paired with a heavier
verification step"):

| Tier | What | Cost | Sees |
|---|---|---|---|
| 0 | motion gate — frame differencing | ~free | every sampled frame |
| 1 | SigLIP2 + rule-deviation health score | ~30–100 fps | what survives the gate |
| 2 | Qwen3-VL-4B with ASK-Hint prompts | ~1–3 fps | ~12% that stage 1 escalates |

Then per-class temporal aggregation turns window verdicts into events with
timestamps.

**No hosted model is in this path.** The PS's sharpest constraint is that
Gemini/NIM/OpenRouter may inform development but cannot be part of what makes
the detector work at runtime. Everything above runs on the T4.

**Run the cells in order.** Cell 2 only *verifies* the mounted dataset — nothing
in this notebook downloads the pack.


## 1 — What's attached

Kaggle's starter cell. It lists everything under `/kaggle/input`, capped at 20
lines because the full pack is 3,200+ files and the stock loop prints one line
each.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# The stock loop prints one line per file. With the full pack attached that is
# 3,200+ lines of scrollback, so the paths are collected and summarised instead.
INPUT_FILES = []
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        INPUT_FILES.append(os.path.join(dirname, filename))

for p in INPUT_FILES[:20]:
    print(p)
if len(INPUT_FILES) > 20:
    print(f"... and {len(INPUT_FILES) - 20} more")
print(f"\n{len(INPUT_FILES)} files under /kaggle/input")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


## 2 — Is the dataset actually there?

**Verification only — nothing here downloads anything.** The pack is attached as
a Kaggle Dataset and mounts read-only.

This cell exists because "the dataset is attached" and "the dataset has videos in
it" are different claims, and the gap between them is silent. It resolves the
mount (including the `datasets/<owner>/<slug>/` shape kagglehub uses), checks all
twelve class folders, the test set and the ground truth, and sets `DATA_OK`.

It also recognises one specific failure by signature: Kaggle's *link a Google
Drive URL* importer cannot authenticate and cannot walk a folder, so pointed at
one it stores a ~360 KB file containing the Drive **web page HTML**. That mounts
happily and holds no data.

In [ ]:
# =============================================================================
# 2 - Is the dataset actually there?
# =============================================================================
# Verification only. Nothing here downloads anything - the pack is attached as a
# Kaggle Dataset and mounts read-only under /kaggle/input.
#
# This cell exists because "the dataset is attached" and "the dataset has videos
# in it" are different claims, and the gap between them is silent: an empty or
# wrong mount indexes to zero videos and the failure only surfaces several cells
# later as a confusing error in the encoder.

from pathlib import Path

# =============================================================================
# WHICH DATASET DOES THIS RUN USE?
# =============================================================================
#   "practice" - the public train/test pack. 34 test videos named T0xx, with
#                ground truth, so cell 9 can score the run and we can iterate.
#   "eval"     - the private evaluation pack. 28 videos named E0xx under
#                L1/L2/L3, NO ground truth (withheld on purpose), so cell 9
#                cannot score anything and cell 10's JSON is the only output.
#
# Only this line changes between the two. Everything downstream reads GT_TEST,
# which cell 3 builds from whichever source this selects - so the pipeline
# itself has no idea which mode it is in, and practice mode behaves exactly as
# it did before this flag existed.
#
# In "eval", attach BOTH datasets on Kaggle:
#     prithvirajgotepatil/ahc-visual-intelligence-eval         <- the videos
#     prithvirajgotepatil/ahc-visual-intelligence-train-test   <- calibration
# Cell 5 measures health_thresh on known-normal TRAINING footage. Without the
# second dataset that threshold stays unset and cells 7/8/11 silently fall back
# to a hard-coded -0.4 that was never measured on anything.
MODE = "practice"    # "practice" | "eval"

assert MODE in ("practice", "eval"), f"MODE must be 'practice' or 'eval', not {MODE!r}"

# The twelve label strings. Scoring compares the string, so these are copied
# exactly from the problem statement and must never be "tidied up".
CLASSES = [
    "normal",
    "traffic_accident",
    "traffic_congestion",
    "stalled_or_broken_down_vehicle",
    "vehicle_blocking_traffic",
    "wrong_way_driving",
    "road_spill_or_debris",
    "waterlogging_or_flood",
    "fire",
    "smoke",
    "fighting_or_violence",
    "loitering_or_suspicious_presence",
]
ANOMALY_CLASSES = [c for c in CLASSES if c != "normal"]

ON_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()


def find_data_roots(max_depth: int = 8) -> list[Path]:
    """Find EVERY directory holding a train/ or test/ - there may be several.

    Two mount shapes have to work, and one of them is not a single tree:

      /kaggle/input/<slug>/                      "Add data"
      /kaggle/input/datasets/<owner>/<slug>/     kagglehub

    ...and when the pack was uploaded as multiple zips, Kaggle extracts each
    archive into its OWN directory named after that archive, so the real mount
    can be five levels down:

      /kaggle/input/datasets/<owner>/<slug>/ahc_part06/Train and Test/train/...

    Google Drive split the pack arbitrarily, so one class folder can have its
    ground_truth.csv in one part and some of its videos in another. There is
    therefore no single "correct root"; the pack is the UNION of these trees,
    which is why this returns a list.

    Depth is not hard-coded, because guessing it kept being wrong - kagglehub
    costs three levels, the archive a fourth, Drive's own folder a fifth. Walk
    and prune instead: a directory containing train/ or test/ IS a root, and
    there is never a reason to descend past one.
    """
    roots: list[Path] = []

    def walk(d: Path, depth: int):
        if depth > max_depth:
            return
        try:
            subdirs = [p for p in d.iterdir() if p.is_dir()]
        except OSError:
            return
        names = {p.name for p in subdirs}
        if "train" in names or "test" in names:
            # resolve() before the dedupe: the bases overlap ("data" and
            # WORK/"data" are one directory), and two Path objects for the same
            # directory are not equal, so an unresolved check counts it twice
            # and every audit number silently doubles.
            r = d.resolve()
            if r not in roots:
                roots.append(r)
            return
        for p in subdirs:
            if p.name in ("videos", "__MACOSX", ".ipynb_checkpoints"):
                continue
            walk(p, depth + 1)

    for base in (Path("/kaggle/input"), WORK / "data", Path("data")):
        if base.exists():
            walk(base, 0)
    return roots


EVAL_LEVEL_DIRS = {"L1", "L2", "L3"}


def find_eval_roots(max_depth: int = 8) -> list[Path]:
    """Find every directory holding the evaluation pack's L1/L2/L3 levels.

    The evaluation pack has a different shape from the train/test pack, and the
    difference is silent rather than loud: no train/, no test/, and no
    ground_truth.csv anywhere - its README says truth is withheld on purpose -
    just L1/, L2/, L3/, each containing videos/ and videos.csv.

    find_data_roots() above matches on a train/ or test/ sibling, so it returns
    [] for this tree. That is not an error anyone sees: it indexes zero videos
    and the run completes having processed nothing. Hence a second finder rather
    than a looser first one - the two shapes stay distinguishable, which is what
    lets both packs be attached at once in eval mode.
    """
    roots: list[Path] = []

    def walk(d: Path, depth: int):
        if depth > max_depth:
            return
        try:
            subdirs = [p for p in d.iterdir() if p.is_dir()]
        except OSError:
            return
        names = {p.name for p in subdirs}
        # every level need not be present - a partial mount is still a root,
        # and saying so beats reporting "no data" for a tree that has videos
        if names & EVAL_LEVEL_DIRS:
            r = d.resolve()
            if r not in roots:
                roots.append(r)
            return
        for p in subdirs:
            if p.name in ("videos", "__MACOSX", ".ipynb_checkpoints"):
                continue
            walk(p, depth + 1)

    for base in (Path("/kaggle/input"), WORK / "eval", Path("eval")):
        if base.exists():
            walk(base, 0)
    return roots


def looks_like_drive_html(p: Path) -> bool:
    """Kaggle's 'link a Google Drive URL' importer cannot authenticate and cannot
    walk a folder - it does a plain GET and stores whatever comes back. Pointed
    at a Drive folder it yields one ~360KB file named after the folder id whose
    content is the Drive *web page*. It mounts perfectly happily and contains no
    data, so name it rather than letting it look like an empty dataset."""
    try:
        if p.is_file() and p.stat().st_size < 2_000_000:
            head = p.read_bytes()[:400].lower()
            return b"<!doctype html" in head and b"google drive" in head
    except OSError:
        pass
    return False


DATA_ROOTS = find_data_roots()
DATA_ROOT = DATA_ROOTS[0] if DATA_ROOTS else (WORK / "data")   # first, for messages
EVAL_ROOTS = find_eval_roots()

print("=" * 74)
print(f"  MODE: {MODE.upper()}" + ("   - private evaluation pack, no ground truth"
                                   if MODE == "eval" else
                                   "   - public train/test pack, scored by cell 9"))
print("=" * 74)

print("attached under /kaggle/input:")
root = Path("/kaggle/input")
if root.exists():
    for d in sorted(p for p in root.iterdir() if p.is_dir()):
        files = [f for f in d.rglob("*") if f.is_file()]
        vids = [f for f in files if f.suffix.lower() == ".mp4"]
        size = sum(f.stat().st_size for f in files)
        print(f"  {d.name:38s} {len(vids):5d} mp4   {size / 1e6:9.1f} MB")
        for f in files:
            if looks_like_drive_html(f):
                print(f"     ! {f.name}")
                print("       This is a Google Drive WEB PAGE, not the dataset.")
                print("       Kaggle's URL importer cannot authenticate or walk a")
                print("       folder, so it saved the HTML it was served.")
else:
    print("  (not running on Kaggle)")

# --- audit, unioned across every root -------------------------------------
videos, gt_files, found_classes = [], [], set()
n_test = 0
for r in DATA_ROOTS:
    videos += list(r.rglob("*.mp4"))
    gt_files += list(r.rglob("ground_truth.csv"))
    if (r / "train").is_dir():
        found_classes |= {p.name for p in (r / "train").iterdir() if p.is_dir()}
    if (r / "test").is_dir():
        n_test += len(list((r / "test").rglob("*.mp4")))
missing, extra = set(CLASSES) - found_classes, found_classes - set(CLASSES)

print(f"\ndata roots: {len(DATA_ROOTS)}")
for r in DATA_ROOTS:
    n = len(list(r.rglob("*.mp4")))
    try:
        shown = r.relative_to("/kaggle/input")
    except ValueError:
        shown = r
    print(f"  {str(shown):58s} {n:5d} mp4")
print(f"\nvideos    : {len(videos)}  ({sum(p.stat().st_size for p in videos) / 1e9:.2f} GB)")
print(f"train     : {len(found_classes)}/12 class folders")
print(f"test      : {n_test} clips")
print(f"ground_truth.csv files: {len(gt_files)}")

# --- the evaluation pack, audited separately -------------------------------
# Counted per level, not in total, because a silently missing level is the
# failure that costs whole marks: L2 and L3 carry 75 of the 100 points between
# just eight videos, so eight absent files is not a rounding error.
n_eval = 0
if EVAL_ROOTS:
    print()
    print(f"eval roots: {len(EVAL_ROOTS)}")
    for r in EVAL_ROOTS:
        try:
            shown = r.relative_to("/kaggle/input")
        except ValueError:
            shown = r
        per = {lv: len(list((r / lv).rglob("*.mp4")))
               for lv in sorted(EVAL_LEVEL_DIRS) if (r / lv).is_dir()}
        n_eval += sum(per.values())
        print(f"  {str(shown):46s} " + "  ".join(f"{k}:{v}" for k, v in per.items()))
    print(f"eval videos: {n_eval}")

if MODE == "eval":
    # An absent ground_truth.csv is the EXPECTED state here, not a fault, so
    # requiring one would reject a perfectly good mount. The practice pack is
    # still wanted - for calibration only - hence a warning, not a stop.
    DATA_OK = n_eval > 0
    if not gt_files:
        print()
        print("  ! the train/test pack is NOT attached. Cell 5 calibrates")
        print("    health_thresh on known-normal TRAINING footage, so without it")
        print("    the threshold stays unset and cells 7/8/11 fall back to a")
        print("    hard-coded -0.4 that was never measured on this data.")
        print("    Add data -> prithvirajgotepatil/ahc-visual-intelligence-train-test")
else:
    DATA_OK = bool(videos) and not missing and n_test > 0 and len(gt_files) > 0

if missing and MODE != "eval":
    print(f"  ! missing class folders: {sorted(missing)}")
if extra and MODE != "eval":
    print(f"  ! unexpected folders: {sorted(extra)}")

if MODE == "eval" and DATA_OK:
    print()
    print(f"DATA OK - {n_eval} evaluation videos."
          + ("" if gt_files else "  (no calibration pack - see the warning above)"))
    print("      Cell 9 cannot score this run: the pack ships no ground truth.")
    print("      Cell 10's arena_submission.json is the output that matters.")
elif MODE == "eval":
    print("""
DATA NOT USABLE. MODE is "eval" but no L1/L2/L3 tree was found.

Attach this one:  Add data -> Your Datasets ->
                  prithvirajgotepatil/ahc-visual-intelligence-eval

It holds the 28-video private evaluation pack (E001-E028, L1 20 / L2 4 / L3 4,
1.33 GB, 47 minutes of footage) and its manifest.json. Attach the train-test
pack alongside it so cell 5 can still calibrate the threshold.
""")
elif DATA_OK:
    print(f"\nDATA OK - all twelve classes, {n_test} test clips, ground truth present.")
    if len(DATA_ROOTS) > 1:
        print(f"      (assembled from {len(DATA_ROOTS)} extracted archives - the pack was")
        print("       uploaded as multiple zips, so Kaggle extracted each into its own")
        print("       directory. Everything below reads the union, so this is fine.)")
else:
    print("""
DATA NOT USABLE. Nothing below will work until a dataset with videos is attached.

Attach this one:  Add data -> Your Datasets ->
                  prithvirajgotepatil/ahc-visual-intelligence-train-test

It holds the full 16.0 GB pack (3,207 clips, all twelve classes, the 34-video
test set and the ground truth). Note it will NOT be found by the Drive-URL
import - that stores a web page, not data, and the earlier
`flytbase-ahc-vis-int` dataset is exactly that failure.
""")


## 3 — Libraries, config, index

Every knob lives in `CFG`, so nothing below carries a magic number. Re-run this
cell after changing one; nothing downstream caches config. It also builds the
ground-truth index (`GT_TRAIN`, `GT_TEST`, `VIDEO_PATHS`) that every later cell
reads.

`USE_FP16` is decided from the GPU rather than assumed. On Kaggle's T4 (sm_75,
real tensor cores) fp16 is right. On a GTX 1650 it is a 3× *slowdown* — TU117
reports capability 7.5 but has the tensor cores fused off, so half precision
falls back to a slow path.

In [ ]:
# =============================================================================
# 3 - Libraries, config, and the ground-truth index
# =============================================================================
# Every knob lives in CFG so nothing below has a magic number. Re-run this cell
# after changing one; nothing downstream caches config.

import json
import re
import time
from dataclasses import dataclass, field

import torch

RUNS = WORK / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

# Every output carries the timestamp of the run that produced it. Without this
# each run overwrites the last, and once a few are downloaded you are left
# guessing which "predictions_raw (7).json" came from which set of changes -
# which matters here, because comparing runs against each other is how nearly
# every finding in this project was made.
RUN_ID = time.strftime("%Y%m%d-%H%M%S")


def run_path(name: str) -> Path:
    """runs/<stem>_<RUN_ID><suffix>, e.g. runs/predictions_raw_20260906-011530.json"""
    p = Path(name)
    return RUNS / f"{p.stem}_{RUN_ID}{p.suffix}"


def free_cuda(*names: str) -> float:
    """Drop the named globals and hand their VRAM back. Returns GB still in use.

    Re-running a model cell in a notebook rebinds the name but does NOT free the
    old weights - the previous module is still referenced until the rebind
    completes, so for a moment two copies of a 9GB model are resident, and on a
    16GB T4 that is an out-of-memory error rather than a slow moment. Deleting
    the name first makes the load work at the cost of a reload; the callers
    below skip even that when the same model is already in memory.
    """
    import gc
    g = globals()
    for n in names:
        if n in g:
            del g[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        return torch.cuda.memory_allocated() / 1e9
    return 0.0


@dataclass
class Config:
    # a list, not a path: an upload of several zips extracts to several sibling
    # trees on Kaggle, and the pack is their union (see cell 2)
    # In eval mode this holds BOTH packs: the eval tree supplies the videos to
    # answer for, the practice tree supplies the known-normal footage cell 5
    # calibrates on. Indexing them together is why no cell below needs to know
    # which pack a given video came from.
    data_roots: list = field(default_factory=lambda: list(DATA_ROOTS) + list(EVAL_ROOTS))

    # --- sampling -------------------------------------------------------
    sample_fps: float = 2.0        # frames/s pulled off the decoder
    max_side: int = 640            # downscale before anything expensive

    # --- stage 0: motion gate -------------------------------------------
    # 4.2 is the measured median frame-diff over the public test set, so this
    # discards ~50% of frames - the rate Cerberus reports. The first guess was
    # 1.6, which passed 12/12 frames on a normal video: a gate that gates
    # nothing. Re-measure if the encoder or sample_fps changes.
    motion_thresh: float = 4.2     # mean abs frame-diff on a 160x90 gray image
    static_keepalive_sec: float = 4.0   # force a frame through even if nothing moves
    visual_prompt: str = "circle"  # "circle" | "square" | "none"

    # --- stage 1: encoder + rule deviation -------------------------------
    encoder_id: str = "google/siglip2-base-patch16-224"
    topk: int = 5                  # rules summed per frame in health()
    escalate_pct: float = 12.0     # % lowest-health frames sent to stage 2
    health_thresh: float | None = None   # set by calibration in cell 5

    # --- scan floor: guarantee long videos are actually looked at ---------
    # health_thresh is calibrated on 5-30s training clips and then applied to
    # 240-629s test videos from different cameras - an absolute cut taken from
    # one distribution and used on another. Measured result: six of the eight
    # anomalous long videos sent 0-1 windows to the VLM, and those videos carry
    # 75 of the 100 available marks. T027 has four real traffic jams and not one
    # of its 420 surviving frames scored below the threshold, so the VLM never
    # looked at it at all.
    #
    # The floor only ever ADDS windows; escalation is untouched. Turn it off and
    # behaviour is exactly as before.
    scan_floor_enabled: bool = True
    scan_floor_min_video_sec: float = 60.0   # L1 clips are 5-27s, L2/L3 are 240s+,
                                             # so nothing sits near this boundary
    scan_floor_interval_sec: float = 20.0    # median real event is 20s, so a 20s
                                             # stride lands inside any median-or-
                                             # longer event; shorter ones can still
                                             # slip through - tighten this to close
                                             # that gap at proportional GPU cost

    # --- stage 2: small VLM ----------------------------------------------
    # Qwen3-VL-4B, not the 3B this was originally pinned to for the GTX 1650's
    # 4GB VRAM. Irrelevant on a T4 (16GB) - see the long comment in cell 6's
    # load_vlm() for the VRAM math and the organizer-referenced papers that
    # independently validate this exact model for this exact task.
    vlm_id: str = "Qwen/Qwen3-VL-4B-Instruct"
    vlm_frames: int = 4            # frames per escalated window
    vlm_max_new_tokens: int = 160

    # REVERTED to 0 (no widening) after measuring it. The theory was that four
    # consecutive samples at 2fps give the model ~2s, too little to separate a
    # collision from the queue it causes. Widening to 8 frames over 16s produced
    # word-for-word identical descriptions on T025 - "a dense queue of vehicles
    # is stopped or moving very slowly on the left side of the highway", before
    # and after - so the extra span bought nothing.
    #
    # It also cost: a 16s span on T032 pulls more scene into view, the model
    # names whatever is most salient across it, and its verdicts went from
    # loitering 10 / stalled 6 to loitering 6 / stalled 6 / accident 3, flipping
    # a correct video-level class to a wrong one. Class accuracy 0.75 -> 0.50,
    # and +3.1 minutes.
    #
    # Set to 16.0 to re-enable; the code path is kept because the idea is sound
    # for a model that can actually resolve the detail, which is the next knob.
    vlm_span_sec: float = 0.0

    # THE ACTUAL CONSTRAINT, measured after the above failed. The test videos are
    # 1280x720 and max_side downscales them to 640x360, discarding 75% of the
    # pixels. On a drone shot of a highway, two vehicles in contact occupy a few
    # dozen pixels at that scale - the model is not failing to reason about
    # damage, it is being handed frames where the damage is gone.
    #
    # Rather than send every frame at full size, crop to the motion region the
    # gate already located and send THAT at native resolution: the detail lands
    # where the evidence is, and the token cost stays flat. vlm_crop_context is
    # how much of the surroundings to keep - a tight crop of a wreck with no road
    # around it is its own kind of unanswerable.
    # REVERTED to False after measuring it, same as the widening above. The crop
    # worked exactly as designed - a small motion box went from 6% of the frame
    # to 25% of the delivered image at native pixels - and changed nothing that
    # mattered. T025's description came back word-for-word identical to the
    # 640-downscale version, "a dense queue of vehicles is stopped or moving
    # very slowly on the left side of the highway", with one clause reworded.
    #
    # It also cost. Cropping tight on T032 turned a stationary three-wheeler
    # into something the model reads as a collision: loitering verdicts fell
    # 10 to 7 and three traffic_accident appeared, flipping a correct
    # video-level class. Class accuracy 0.75 to 0.50, plus 2.2 minutes.
    #
    # Taken with the widening result this is a clean three-way control. Same
    # footage, same question, and the model returns the same sentence whether it
    # gets 2 seconds or 16, 640px or native pixels on the region of interest.
    # It is not being starved of evidence - that is genuinely what it sees.
    #
    # Both code paths are kept and defaulted off. They are the right levers for
    # a model that can use them, which is now the open question rather than
    # something to keep tuning here.
    vlm_crop_to_motion: bool = False
    vlm_crop_context: float = 3.0   # multiple of the motion box to include
    vlm_crop_min_px: int = 320      # never crop below this, small boxes need room

    # --- temporal aggregation --------------------------------------------
    enter_conf: float = 0.55       # stage-2 confidence to open an event
    exit_conf: float = 0.35        # ...and to close it (hysteresis)
    merge_gap_sec: float = 3.0     # bridge two events of the same class

    # The only declared constant left in the extent path. Cell 7 removed the 15s
    # floor, the 180s cap and both merge-gap constants; what remains is a
    # symmetric quantisation allowance, because frames land on a 1/sample_fps
    # grid and a real boundary can sit one interval outside the window that
    # caught it. Larger helps a straddled boundary and hurts a very short event
    # (IoU falls once the prediction outgrows the truth), so it lives here to be
    # swept offline against stored window_verdicts rather than re-run on a GPU.
    extent_buffer_sec: float = 2.0

    # --- the learned prior (cell 5b) --------------------------------------
    # All three chosen by sweeping the cached training embeddings offline, which
    # costs seconds and no GPU. Held out 559 clips, 25% of the capped set:
    #
    #   C          top-1   top-5   recall@0.90   specificity@0.90
    #   1          0.673   0.969      0.849           0.893
    #   10         0.717   0.977      0.903           0.933
    #   100        0.739   0.983      0.930           0.947   <- chosen
    #   300        0.741   0.981      0.938           0.933
    #
    # Less regularisation is better here, which is what 768 well-conditioned
    # frozen features and 1,676 training rows should predict.
    probe_C: float = 100.0

    # Escalate a window when the probe puts P(anomalous) at or above this.
    # 0.90 buys recall 0.930 at specificity 0.947 - it replaces a health
    # threshold measured to be INVERTED on three of four test videos.
    probe_escalate_p: float = 0.90

    # ...and the bar for the probe to CONTRADICT stage 2. Started at 0.95, the
    # point where held-out training clips gave zero false positives. Real
    # footage then said that was needlessly strict:
    #
    #   video   highest probe score OUTSIDE any real event
    #   T030    0.582   <- a wholly normal video, the thing we must not flag
    #   T026    0.670
    #   T025    0.858
    #
    # T030 is the one that matters and it tops out at 0.582, so 0.90 keeps a
    # 0.32 margin on the only video where a false alarm is possible. Dropping
    # from 0.95 to 0.90 fires on two more of T025's six real accidents, whose
    # in-event scores run 0.748 to 0.977.
    #
    # The windows this fires on OUTSIDE an event are almost all T031 and T032,
    # where the probe believes the whole video is anomalous. Those are extent
    # errors inside videos we already flag, not false alarms on normal ones -
    # a different and much cheaper kind of wrong.
    probe_override_p: float = 0.90
    probe_override_enabled: bool = True


CFG = Config()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
CAP = torch.cuda.get_device_capability(0) if DEVICE == "cuda" else (0, 0)
# T4 (sm_75, real tensor cores) wants fp16. A GTX 1650 is TU117 - capability 7.5
# but with the tensor cores fused off, so fp16 has no fast path and measures 3x
# SLOWER than fp32 (11 vs 33 fps for SigLIP2-base). Decide from the device.
USE_FP16 = DEVICE == "cuda" and not any(x in GPU_NAME for x in ("1650", "1660"))
DTYPE = torch.float16 if USE_FP16 else torch.float32

print(f"device : {DEVICE}  {GPU_NAME}  sm_{CAP[0]}{CAP[1]}")
print(f"dtype  : {DTYPE}")
print(f"torch  : {torch.__version__}")

# --- index -------------------------------------------------------------------
# video_id in the CSVs is the filename stem, so one map over every root gives the
# whole pack and no other cell has to know the layout. Unioning is what makes a
# split upload (8 sibling trees) behave identically to a single tree.
#
# DO NOT switch this to videos.csv's `filename` column. It holds a path relative
# to the CSV ("videos/T001.mp4"), and on a split upload every CSV sits in the
# first archive while the videos are spread across all eight - so 2,771 of 3,207
# rows (86%) point at files that are not there. Measured, not hypothetical. A
# stem is location-independent; a relative path is not. ground_truth.csv has no
# path column at all, which is the join we actually rely on.
VIDEO_PATHS = {}
for _root in CFG.data_roots:
    for _p in _root.rglob("*.mp4"):
        VIDEO_PATHS.setdefault(_p.stem, _p)


def load_ground_truth(split: str) -> pd.DataFrame:
    """Concatenate every ground_truth.csv under train/ or test/, across all roots.

    train/ has one per class folder; test/ has a single one. Both share the
    schema, so one loader serves either, and `path` is added for convenience.
    Rows are de-duplicated because a split upload can surface the same CSV twice.
    """
    frames = []
    for root in CFG.data_roots:
        base = root / split
        if not base.exists():
            continue
        for csv in sorted(base.rglob("ground_truth.csv")):
            df = pd.read_csv(csv)
            df["source_csv"] = str(csv)
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    gt = pd.concat(frames, ignore_index=True)
    key = [c for c in ("video_id", "level", "class_name", "start_time_sec",
                       "end_time_sec") if c in gt]
    gt = gt.drop_duplicates(subset=key).reset_index(drop=True)
    for col in ("start_time_sec", "end_time_sec"):
        if col in gt:
            gt[col] = pd.to_numeric(gt[col], errors="coerce")
    gt["path"] = gt["video_id"].astype(str).map(VIDEO_PATHS)
    return gt


EVAL_MANIFEST_PATH = None
for _r in EVAL_ROOTS:
    _m = _r / "manifest.json"
    if _m.exists():
        EVAL_MANIFEST_PATH = _m
        break


def load_eval_index() -> pd.DataFrame:
    """Build a GT_TEST-shaped frame for the evaluation pack.

    This is the hinge the whole flag turns on. It returns the SAME columns
    load_ground_truth returns - video_id, level, path, and the truth columns
    class_name / start_time_sec / end_time_sec / is_anomaly - except the truth
    columns are empty, because the pack ships none.

    Keeping the shape identical is deliberate. Cells 8, 10 and 11 all consume
    GT_TEST, and every one of them keeps working with no branch of its own; the
    alternative was a MODE check in five places, each of which could drift. The
    truth columns are present-but-NaN rather than absent so that cell 9 can ask
    "is there truth here?" and get a clean answer instead of a KeyError.

    Level comes from the manifest, not the folder name, because the manifest is
    what the arena scores against - and duration_sec comes from it too, which is
    more authoritative than our own decode.
    """
    rows = []
    if EVAL_MANIFEST_PATH is not None:
        man = json.loads(EVAL_MANIFEST_PATH.read_text())
        for v in man.get("videos", man if isinstance(man, list) else []):
            rows.append({"video_id": v["video_id"],
                         "level": int(v.get("level", v.get("difficulty", 1))),
                         "duration_sec": float(v.get("duration_sec", 0)) or None})
    else:
        # No manifest: fall back to the tree itself. Levels come from the folder
        # name here, which is the best available and matches how it was shipped.
        print("  ! no manifest.json in the eval pack - falling back to the "
              "L1/L2/L3 folder names for levels")
        for r in EVAL_ROOTS:
            for lv in sorted(EVAL_LEVEL_DIRS):
                for mp4 in sorted((r / lv).rglob("*.mp4")) if (r / lv).is_dir() else []:
                    rows.append({"video_id": mp4.stem, "level": int(lv[1]),
                                 "duration_sec": None})
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows).drop_duplicates("video_id").reset_index(drop=True)
    for col in ("class_name", "start_time_sec", "end_time_sec", "is_anomaly"):
        df[col] = pd.NA
    df["path"] = df["video_id"].map(VIDEO_PATHS)
    return df


GT_TRAIN = load_ground_truth("train")
GT_TEST = load_eval_index() if MODE == "eval" else load_ground_truth("test")

# Truth present or not is asked once, here, and answered everywhere else by
# reading this - cells 9 and 11 must not re-derive it from MODE, or the two
# could disagree.
HAS_TRUTH = (not GT_TEST.empty) and GT_TEST["class_name"].notna().any()

print(f"\nindexed {len(VIDEO_PATHS)} videos across {len(CFG.data_roots)} root(s)")
for name, gt in (("train", GT_TRAIN), ("test", GT_TEST)):
    if gt.empty:
        print(f"  {name}: no ground_truth.csv found")
        continue
    missing = int(gt["path"].isna().sum())
    print(f"  {name}: {len(gt)} rows, {gt['video_id'].nunique()} videos"
          f"{f', {missing} rows with no matching mp4' if missing else ''}")
    unknown = set(gt.get("class_name", pd.Series(dtype=str)).dropna()) - set(CLASSES)
    if unknown:
        print(f"  ! labels outside the twelve: {sorted(unknown)}")

if MODE == "eval":
    print()
    print(f"EVAL SET: {len(GT_TEST)} videos to answer for"
          + (f", manifest {EVAL_MANIFEST_PATH.name}" if EVAL_MANIFEST_PATH else ""))
    if not GT_TEST.empty:
        for _lv, _n in sorted(GT_TEST["level"].value_counts().items()):
            _d = GT_TEST[GT_TEST["level"] == _lv]["duration_sec"]
            print(f"  L{_lv}: {_n:2d} videos, {_d.sum() / 60:5.1f} min")
        _miss = GT_TEST[GT_TEST["path"].isna()]["video_id"].tolist()
        if _miss:
            print(f"  ! {len(_miss)} manifest video(s) with no mp4 on disk: "
                  f"{_miss[:8]}{' ...' if len(_miss) > 8 else ''}")
    print(f"  ground truth: {'present' if HAS_TRUTH else 'ABSENT (cell 9 will skip)'}")


## 4 — Stage 0: sampling and the motion gate

One frame-differencing pass does two jobs, as in Cerberus: decide whether a
frame is worth encoding, and locate the moving region so a red circle can be
drawn on it as a visual prompt.

**The keepalive is ours, not the paper's, and it matters.** Cerberus gates on
motion because its anomalies are motion events. Three of our twelve labels are
not: `waterlogging_or_flood` and `road_spill_or_debris` are static conditions,
and `stalled_or_broken_down_vehicle` is *defined* by the absence of motion. A
pure motion gate discards exactly their evidence, so one frame is forced through
every `static_keepalive_sec` regardless of score.

In [ ]:
# =============================================================================
# 3 - Frame sampling, motion gate, visual prompting
# =============================================================================
# Stage 0 of the cascade. One frame-differencing computation does two jobs, as
# in Cerberus: it decides whether a frame is worth encoding at all, and it
# locates the moving region so we can draw a visual prompt on it.

import cv2
from PIL import Image


def iter_sampled_frames(path, target_fps: float, max_side: int = 640):
    """Yield (t_seconds, bgr_frame) at roughly target_fps.

    grab() advances the decoder without colour-converting or copying; retrieve()
    is only paid on frames we keep. At 2 fps off 25 fps source that is ~12x less
    work than read()-ing everything. Seeking per-sample with CAP_PROP_POS_FRAMES
    would be worse still - every seek forces a keyframe jump and re-decode.
    """
    cap = cv2.VideoCapture(str(path))
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    if not src_fps or src_fps != src_fps or src_fps <= 0:   # 0, or NaN
        src_fps = 25.0
    stride = max(1, int(round(src_fps / target_fps)))
    i = 0
    try:
        while True:
            if not cap.grab():
                break
            if i % stride == 0:
                ok, frame = cap.retrieve()
                if ok and frame is not None:
                    h, w = frame.shape[:2]
                    if max(h, w) > max_side:
                        s = max_side / max(h, w)
                        frame = cv2.resize(frame, (int(w * s), int(h * s)))
                    yield i / src_fps, frame
            i += 1
    finally:
        cap.release()


def video_duration(path) -> float:
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    n = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    cap.release()
    return float(n / fps) if fps > 0 else 0.0


class MotionGate:
    """Frame differencing on a 160x90 grayscale thumbnail.

    The keepalive is not in the paper and matters a lot here. Cerberus gates on
    motion because its anomalies are motion events; three of our twelve labels
    are not. waterlogging_or_flood and road_spill_or_debris are static
    conditions, and stalled_or_broken_down_vehicle is *defined* by the absence of
    motion once a vehicle has been still long enough. A pure motion gate would
    discard precisely the frames that carry their evidence, so we force one
    frame through every static_keepalive_sec regardless of score.
    """

    def __init__(self, cfg):
        self.thresh = cfg.motion_thresh
        self.keepalive = cfg.static_keepalive_sec
        self.prev = None
        self.last_pass = -1e9

    def __call__(self, t: float, frame):
        small = cv2.cvtColor(cv2.resize(frame, (160, 90)), cv2.COLOR_BGR2GRAY)
        if self.prev is None:
            self.prev, self.last_pass = small, t
            return True, 0.0, None, "first"

        diff = cv2.absdiff(small, self.prev)
        self.prev = small
        score = float(diff.mean())

        moving = score >= self.thresh
        stale = (t - self.last_pass) >= self.keepalive
        if not (moving or stale):
            return False, score, None, "skipped"

        self.last_pass = t
        box = self._largest_region(diff, frame.shape) if moving else None
        return True, score, box, ("motion" if moving else "keepalive")

    @staticmethod
    def _largest_region(diff, shape):
        """Bounding box of the biggest moving blob, in full-frame coordinates."""
        _, mask = cv2.threshold(diff, 18, 255, cv2.THRESH_BINARY)
        mask = cv2.dilate(mask, np.ones((5, 5), np.uint8), iterations=2)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts:
            return None
        x, y, w, h = cv2.boundingRect(max(cnts, key=cv2.contourArea))
        if w * h < 12:                      # noise, not a subject
            return None
        H, W = shape[:2]
        sx, sy = W / 160.0, H / 90.0
        return int(x * sx), int(y * sy), int(w * sx), int(h * sy)


def draw_visual_prompt(frame, box, style: str = "circle"):
    """Overlay a red marker on the moving region.

    Cerberus finds circles pull VLM attention harder (better recall) while
    squares admit less background (better precision), and picks between them by
    motion scale. We expose the choice and default to the recall-favouring one,
    because stage 2 is the thing that can say no - a miss here is unrecoverable.
    """
    if box is None or style == "none":
        return frame
    out = frame.copy()
    x, y, w, h = box
    if style == "square":
        cv2.rectangle(out, (x, y), (x + w, y + h), (0, 0, 255), 3)
    else:
        cx, cy = x + w // 2, y + h // 2
        cv2.circle(out, (cx, cy), max(18, int(0.6 * max(w, h))), (0, 0, 255), 3)
    return out


def to_pil(frame):
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))


def native_crop(cap, t: float, box, small_wh, cfg=None):
    """Re-read frame `t` at NATIVE resolution and crop around the motion box.

    Stage 0 stores frames already downscaled to max_side, because that is all
    the motion gate and the encoder need. Stage 2 needs more: the test videos
    are 1280x720 and max_side 640 discards 75% of the pixels, so on a drone shot
    of a highway two vehicles in contact are a few dozen pixels by the time the
    VLM sees them. Measured consequence - given 2s or 16s of that footage the
    model writes the same sentence, "a dense queue of vehicles is stopped or
    moving very slowly", because the damage that would make it an accident is
    no longer in the image to describe.

    So spend a seek on the frames stage 2 actually looks at, and crop to where
    the motion was, keeping cfg.vlm_crop_context times the box for surroundings
    - a tight crop of a wreck with no road around it is its own kind of
    unanswerable. Downscaling only happens if the crop is still larger than
    max_side, so a modest box comes back at full detail and the token cost is
    unchanged.

    Returns None whenever anything is missing, so the caller keeps the frame it
    already has.
    """
    cfg = cfg or CFG
    if cap is None or box is None or not getattr(cfg, "vlm_crop_to_motion", False):
        return None
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, int(t * fps)))
    ok, frame = cap.read()
    if not ok or frame is None:
        return None

    H, W = frame.shape[:2]
    sw, sh = small_wh
    if not sw or not sh:
        return None
    # box is in the DOWNSCALED frame's pixel space; map it back up
    fx, fy = W / float(sw), H / float(sh)
    x, y, w, h = box
    cx, cy = (x + w / 2) * fx, (y + h / 2) * fy
    half_w = max(w * fx * cfg.vlm_crop_context, cfg.vlm_crop_min_px) / 2
    half_h = max(h * fy * cfg.vlm_crop_context, cfg.vlm_crop_min_px) / 2
    x0, x1 = int(max(0, cx - half_w)), int(min(W, cx + half_w))
    y0, y1 = int(max(0, cy - half_h)), int(min(H, cy + half_h))
    if x1 - x0 < 32 or y1 - y0 < 32:
        return None
    crop = frame[y0:y1, x0:x1]
    ch, cw = crop.shape[:2]
    if max(ch, cw) > cfg.max_side:
        s = cfg.max_side / max(ch, cw)
        crop = cv2.resize(crop, (int(cw * s), int(ch * s)))
    return crop


def sample_video(path, cfg=None):
    """Full stage-0 pass. Returns the frames that survive the gate."""
    cfg = cfg or CFG
    gate = MotionGate(cfg)
    kept, n_seen = [], 0
    for t, frame in iter_sampled_frames(path, cfg.sample_fps, cfg.max_side):
        n_seen += 1
        passed, score, box, why = gate(t, frame)
        if passed:
            kept.append({"t": t, "frame": frame, "motion": score,
                         "box": box, "why": why})
    return kept, n_seen


# --- smoke test on one video --------------------------------------------------
_probe = next(iter(VIDEO_PATHS.values()), None)
if _probe is not None:
    _t0 = time.time()
    _kept, _seen = sample_video(_probe)
    _dt = time.time() - _t0
    _why = pd.Series([k["why"] for k in _kept]).value_counts().to_dict()
    print(f"{_probe.name}: {_seen} sampled -> {len(_kept)} kept "
          f"({100 * len(_kept) / max(_seen, 1):.0f}%) in {_dt:.1f}s   {_why}")
    print(f"stage-0 throughput: {_seen / max(_dt, 1e-6):.0f} sampled-frames/s")
else:
    print("no videos indexed yet - run cells 2 and 3 first")


## 5 — Stage 1: rule-deviation scoring

`health(x) = Σ_{c ∈ topk(x)} w_c · sim(x, c)`, with `w = +1` for normal rules and
`w = −1` for perturbed action labels. Escalate when health is low.

Two reasons this beats the obvious approach of listing anomalies and matching
against them. First, Cerberus measured that: prompting an LLM for possible
anomalies gave **27.13% recall on ShanghaiTech, 21.81% on NWPU** — enumeration
misses most of what happens. Second, Alert-CLIP shows CLIP's normal and abnormal
text embeddings are *entangled*, so asking it to compare "a normal street"
against "an anomalous street" is measurably unreliable. Here we never ask that
question; we ask which concrete descriptions the frame is nearest and let the
signs do the work.

The threshold is **calibrated on known-normal training footage**, not guessed,
which turns `escalate_pct` into a compute budget you can reason about.

In [ ]:
# =============================================================================
# 4 - Stage 1: rule-deviation scoring over frame embeddings
# =============================================================================
# The always-on tier. No training. The key idea, from Cerberus: do NOT enumerate
# anomalies and match against them. Prompting an LLM for a list of possible
# anomalies and matching gave them 27.13% recall on ShanghaiTech and 21.81% on
# NWPU - enumeration misses most of what actually happens.
#
# Instead score each frame against a pool of *normal* rules (weight +1) and
# *perturbed* atomic action labels (weight -1), take the top-k by cosine
# similarity and sum:
#
#     health(x) = sum_{c in topk(x)} w_c * sim(x, c)
#     escalate  <=>  health(x) < threshold
#
# This also sidesteps CLIP's normal/abnormal text entanglement (Alert-CLIP,
# CVPR 2026): we never ask CLIP to compare "a normal street" against "an
# anomalous street", which it is measurably bad at. We ask which of many
# concrete descriptions the frame is nearest, and let the signs do the work.

import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor

# --- the rule pool ------------------------------------------------------------
# Positive rules: what routine footage in these domains actually looks like.
# Deliberately behavioural and specific - "traffic flowing" not "a road".
NORMAL_RULES = [
    "vehicles driving steadily along a road in the same direction",
    "cars moving at a constant speed on a highway",
    "traffic flowing smoothly through an intersection",
    "vehicles waiting in an orderly queue at a red traffic light",
    "cars parked in marked bays in a car park",
    "a pedestrian walking along a pavement",
    "people walking calmly across a crossing",
    "a person waiting at a bus stop",
    "a cyclist riding along the side of the road",
    "an empty road with no vehicles",
    "an empty street at night lit by street lamps",
    "a quiet campus walkway with a few people walking",
    "people standing and talking in a group",
    "a delivery van stopped briefly at the kerb",
    "a motorcycle riding in its lane",
    "a bus stopping at a designated bus stay",
    "clear dry road surface with lane markings visible",
    "an aerial view of a city street with normal traffic",
    "a drone view of rooftops and roads with light traffic",
    "a roundabout with vehicles circulating normally",
    "a toll booth with cars passing through in turn",
    "a footpath beside a road with occasional pedestrians",
    "an open park area with people walking",
    "a railway platform with passengers waiting",
    "vehicles changing lanes normally in flowing traffic",
    "a construction site with normal work in progress",
    "a car indicating and turning at a junction",
    "trees and buildings beside a road",
    "an overhead view of a car park with stationary parked cars",
    "night traffic with headlights moving steadily",
]

# Negative rules: atomic action labels in the style of Moments in Time, which
# Cerberus uses as perturbed negatives. This is a working subset chosen for our
# twelve labels; the full MiT vocabulary is 339 and adding more is cheap - it is
# one forward pass of the text tower, done once.
PERTURBED_ACTIONS = [
    "crashing", "colliding", "overturning", "flipping", "derailing",
    "burning", "flaming", "smoking", "exploding", "erupting",
    "flooding", "submerging", "overflowing", "leaking", "spilling",
    "punching", "kicking", "fighting", "wrestling", "shoving",
    "falling", "collapsing", "stumbling", "tripping", "slipping",
    "running away", "fleeing", "chasing", "panicking", "scattering",
    "crowding", "stampeding", "swarming", "queueing motionless",
    "loitering", "lurking", "trespassing", "climbing a fence",
    "breaking", "smashing", "vandalising", "stealing",
    "skidding", "swerving", "reversing into traffic", "driving against traffic",
    "blocking the road", "stalling", "breaking down", "stranded",
    "towing", "rescuing", "evacuating", "carrying an injured person",
    "crying", "shouting", "screaming", "arguing",
    "collapsed on the ground", "lying motionless on the road",
    "debris scattered on the road", "smoke rising", "water covering the road",
]

RULES = NORMAL_RULES + PERTURBED_ACTIONS
RULE_W = torch.tensor([1.0] * len(NORMAL_RULES) + [-1.0] * len(PERTURBED_ACTIONS))

# --- encoder ------------------------------------------------------------------
# Re-running this cell must not load a second copy alongside the first. Reuse
# when the same encoder is already resident, and free the old weights before
# loading when it is not - see free_cuda() in cell 3 for why the rebind alone
# is not enough.
if globals().get("_ENCODER_ID") == CFG.encoder_id and "encoder" in globals():
    print(f"encoder: {CFG.encoder_id} already loaded, reusing "
          f"({torch.cuda.memory_allocated() / 1e9:.2f} GB in use)"
          if DEVICE == "cuda" else
          f"encoder: {CFG.encoder_id} already loaded, reusing")
else:
    free_cuda("encoder", "processor")
    try:
        processor = AutoProcessor.from_pretrained(CFG.encoder_id)
        encoder = AutoModel.from_pretrained(CFG.encoder_id, torch_dtype=DTYPE)
    except Exception as e:
        print(f"{CFG.encoder_id} unavailable ({str(e).splitlines()[0][:120]});"
              " falling back to CLIP-B/16")
        CFG.encoder_id = "openai/clip-vit-base-patch16"
        processor = AutoProcessor.from_pretrained(CFG.encoder_id)
        encoder = AutoModel.from_pretrained(CFG.encoder_id, torch_dtype=DTYPE)
    encoder = encoder.to(DEVICE).eval()
    _ENCODER_ID = CFG.encoder_id
IS_SIGLIP = "siglip" in CFG.encoder_id.lower()
print(f"encoder: {CFG.encoder_id}  {sum(p.numel() for p in encoder.parameters()) / 1e6:.0f}M params")


def _as_tensor(out):
    """get_*_features returns a bare tensor on some transformers versions and a
    BaseModelOutputWithPooling on others. Kaggle's image pins its own version and
    it will not be the one this was written against, so normalise rather than
    depend on either."""
    if torch.is_tensor(out):
        return out
    for attr in ("pooler_output", "image_embeds", "text_embeds", "last_hidden_state"):
        v = getattr(out, attr, None)
        if v is not None:
            return v.mean(1) if v.ndim == 3 else v
    raise TypeError(f"cannot get embeddings from {type(out)}")


@torch.no_grad()
def embed_texts(texts, batch=64):
    """SigLIP requires padding='max_length' - it was trained with a fixed 64-token
    context and dynamic padding silently degrades the embeddings. CLIP does not
    care. Getting this wrong produces a pipeline that runs and scores noise."""
    out = []
    for i in range(0, len(texts), batch):
        chunk = texts[i:i + batch]
        kw = {"padding": "max_length", "max_length": 64} if IS_SIGLIP else {"padding": True}
        inp = processor(text=chunk, return_tensors="pt", truncation=True, **kw)
        inp = {k: v.to(DEVICE) for k, v in inp.items()}
        feats = _as_tensor(encoder.get_text_features(**inp))
        out.append(F.normalize(feats.float(), dim=-1).cpu())
    return torch.cat(out)


@torch.no_grad()
def embed_images(pil_images, batch=32):
    _t0 = time.time()
    out = []
    for i in range(0, len(pil_images), batch):
        inp = processor(images=pil_images[i:i + batch], return_tensors="pt")
        pv = inp["pixel_values"].to(DEVICE, dtype=DTYPE)
        feats = _as_tensor(encoder.get_image_features(pixel_values=pv))
        out.append(F.normalize(feats.float(), dim=-1).cpu())
    _log_call("siglip2-encoder", (time.time() - _t0) * 1000)
    return torch.cat(out)


# Per-model call timings, for the arena submission's runtime_metadata.
# model_runtimes (call_count/total/avg/p50/p95/max per video). process_video()
# in cell 8 snapshots this before and after each video to get per-video stats.
CALL_LOG: dict[str, list[float]] = {}


def _log_call(model_name: str, elapsed_ms: float) -> None:
    CALL_LOG.setdefault(model_name, []).append(elapsed_ms)


RULE_EMB = embed_texts(RULES)
print(f"rule pool: {len(NORMAL_RULES)} normal (+1), {len(PERTURBED_ACTIONS)} perturbed (-1)")


def health(img_emb: torch.Tensor, topk: int | None = None) -> torch.Tensor:
    """health(x) = sum over the top-k nearest rules of w_c * sim(x, c).

    Low health means the frame's nearest neighbours in the rule pool are mostly
    perturbed actions - i.e. it does not look like anything we called normal.
    """
    topk = topk or CFG.topk
    sim = img_emb @ RULE_EMB.T                      # (N, R), both L2-normalised
    top_sim, top_idx = sim.topk(topk, dim=-1)
    return (top_sim * RULE_W[top_idx]).sum(-1)


def stage1_video(path, cfg=None):
    """Stage 0 + stage 1 over one video. Returns per-kept-frame records."""
    cfg = cfg or CFG
    kept, n_seen = sample_video(path, cfg)
    if not kept:
        return [], n_seen
    imgs = [to_pil(draw_visual_prompt(k["frame"], k["box"], cfg.visual_prompt)) for k in kept]
    emb = embed_images(imgs)
    h = health(emb)
    # Keep the embedding, not just the scalar derived from it. The probe in cell
    # 5b needs 8 frames mean-pooled over 16s, and these are exactly those frames
    # already encoded - so retaining them makes probe scoring cost ZERO extra
    # GPU rather than a second pass. About 2.7 MB for the longest test video
    # (870 kept frames x 768 floats), which is nothing against the frames
    # themselves already held in `kept`.
    _e = emb.detach().float().cpu().numpy()
    for k, hv, ev in zip(kept, h.tolist(), _e):
        k["health"] = hv
        k["emb"] = ev
    return kept, n_seen


# --- calibrate the threshold on normal training footage -----------------------
# Picking a health threshold by eye is guesswork; the distribution differs per
# encoder and per rule pool. Instead measure health on footage we KNOW is normal
# and set the cut so escalate_pct of it escalates. That makes escalate_pct a
# compute budget - "stage 2 runs on ~12% of frames" - rather than a magic number.
def calibrate(n_videos: int = 12, cfg=None):
    cfg = cfg or CFG
    if GT_TRAIN.empty:
        print("no training ground truth - leaving health_thresh unset")
        return None
    normals = (GT_TRAIN[(GT_TRAIN["class_name"] == "normal") & GT_TRAIN["path"].notna()]
               .drop_duplicates("video_id").head(n_videos))
    if normals.empty:
        print("no normal videos found - leaving health_thresh unset")
        return None

    scores = []
    t0 = time.time()
    for _, row in normals.iterrows():
        kept, _ = stage1_video(row["path"], cfg)
        scores += [k["health"] for k in kept]
    if not scores:
        return None

    arr = np.array(scores)
    thr = float(np.percentile(arr, cfg.escalate_pct))
    cfg.health_thresh = thr
    print(f"calibrated on {len(normals)} normal videos, {len(arr)} frames, "
          f"{time.time() - t0:.0f}s")
    print(f"  health: mean {arr.mean():.3f}  p1 {np.percentile(arr, 1):.3f}  "
          f"p50 {np.percentile(arr, 50):.3f}  p99 {np.percentile(arr, 99):.3f}")
    print(f"  health_thresh = {thr:.4f}  (escalates the lowest "
          f"{cfg.escalate_pct:.0f}% of normal frames)")
    return thr


calibrate()


## 5b — A learned prior, because the written one is inverted

The health score above is the foundation under escalation, clustering and
extent measurement. Measured against ground truth, it is **flat or backwards**
on three of four test videos — anomalous frames sit at the *65th percentile* of
their own video's health. A congested road looks like a road; a loiterer looks
like a person.

So this cell spends the **3,173 labelled training clips** we have so far used to
compute exactly one number. Each clip is embedded once — 8 frames spanning 16s,
centred on the labelled event — and a plain multinomial logistic regression is
fitted over the frozen SigLIP2 vectors. The encoder stays frozen and zero-shot:
no backprop, no fine-tuning, no GPU for the fit. Ninety-three English sentences
are replaced by coefficients learned from examples.

**8 frames over 16s, not 4 over 2s**, on both sides. Our inference window is ~2s
while the median real event is 20s, so training wide and predicting narrow would
rebuild the exact mismatch this is meant to remove.

Embeddings are cached (`WORK`, then any attached dataset), because the expensive
step is pixels→vectors and the step worth iterating on is the classifier over
them. Nothing downstream consumes the probe yet — that decision waits on the
held-out numbers printed here.

In [ ]:
# =============================================================================
# 5b - A learned prior, because the written one is inverted
# =============================================================================
# Cell 5 scores a frame by comparing it to 30 hand-written normal rules and 63
# perturbed actions. Measured on the five-video run, comparing health INSIDE a
# real ground-truth event against health OUTSIDE it in the same video:
#
#     T026  inside +0.134  outside +0.608   separation +0.474   works
#     T031  inside -0.085  outside -0.092   separation -0.007   flat
#     T032  inside +0.531  outside +0.415   separation -0.116   INVERTED
#     T025  inside +0.150  outside -0.097   separation -0.247   INVERTED
#
# On three of four videos the anomalous frames are as healthy as or HEALTHIER
# than the rest of the video - they sit at the 65th percentile of their own
# video's health. A congested road looks like a road; a loiterer looks like a
# person. Only T026's road spill, a visible appearance change, separates.
#
# That one signal drives escalation, clustering and extent measurement, so it is
# the foundation under most of the pipeline, and it is upside down.
#
# We also have 3,173 labelled training clips that we have so far used to compute
# exactly one number (health_thresh). This cell spends them properly: embed each
# clip once, then fit a plain multinomial logistic regression over the frozen
# SigLIP2 embeddings. The encoder stays frozen and zero-shot - no backprop, no
# fine-tuning, no GPU for the fit itself. What changes is that 93 English
# sentences are replaced by coefficients learned from labelled examples.
#
# THIS CELL ONLY BUILDS AND MEASURES THE PROBE. Nothing downstream consumes it
# yet, deliberately: how it should be used depends on how good it turns out to
# be, and wiring it in before measuring it would make that unanswerable.

import numpy as np

PROBE_FRAMES = 8            # frames per training example
PROBE_SPAN_SEC = 16.0       # ...spanning this much video
PROBE_MAX_PER_CLASS = 300   # cap: normal has 973 clips and fire has 77
PROBE_CACHE_NAME = f"train_emb_{PROBE_FRAMES}x{int(PROBE_SPAN_SEC)}s.npz"

# 8 frames over 16s, not 4 over 2s, and the reason is the whole point of the
# cell. Our inference window is ~2s wide while the median real event is 20s, so
# a probe trained on wide clips and applied to 2s windows would rebuild the
# train/test mismatch it exists to remove. Both sides use this shape.


# --- clips the organisers reissued labels for, and we now exclude ------------
# data/train/wrong_way_driving/ holds 164 clips, all labelled wrong_way_driving.
# The reissued ground_truth_corrected_v2.csv covers exactly those 164 and calls
# 108 of them NORMAL - two thirds of the class was mislabelled. Three options,
# all measured by refitting the cached embeddings (no re-embed, seconds):
#
#   handling of the 108        top-1   spec@0.90   times wrong_way predicted
#   keep as wrong_way (before) 0.739     0.947              40
#   relabel to normal          0.737     0.912              31
#   DROP them                  0.767     0.947              19   <- chosen
#
# Relabelling looks like the obvious fix and is the worst of the three: it
# pours 108 ambiguous road scenes into `normal`, which muddies the one class
# whose precision protects our false-alarm record - specificity falls from
# 0.947 to 0.912. Dropping them is better on every axis, and it halves how
# often wrong_way_driving is predicted at all, which is the failure that costs
# us most: on T025 the probe localises five real accidents at IoU 0.8+ and
# calls every one of them wrong_way_driving.
#
# The honest reading is that these 108 are disputed rather than known. The
# organisers changed their mind about them once, so asserting either label
# trains on a guess; excluding them asserts nothing.
PROBE_DISPUTED_IDS = frozenset([
    "TR00001", "TR00002", "TR00004", "TR00005", "TR00006", "TR00007",
    "TR00009", "TR00010", "TR00012", "TR00013", "TR00015", "TR00016",
    "TR00017", "TR00019", "TR00020", "TR00021", "TR00022", "TR00027",
    "TR00029", "TR00031", "TR00032", "TR00033", "TR00037", "TR00040",
    "TR00043", "TR00044", "TR00045", "TR00047", "TR00049", "TR00052",
    "TR00056", "TR00058", "TR00059", "TR00062", "TR00064", "TR00065",
    "TR00066", "TR00068", "TR00069", "TR00072", "TR00073", "TR00074",
    "TR00075", "TR00076", "TR00077", "TR00078", "TR00079", "TR00080",
    "TR00081", "TR00082", "TR00083", "TR00084", "TR00085", "TR00087",
    "TR00088", "TR00089", "TR00092", "TR00093", "TR00099", "TR00100",
    "TR02761", "TR02771", "TR02790", "TR02798", "TR02806", "TR02814",
    "TR02822", "TR02837", "TR02844", "TR02851", "TR02872", "TR02879",
    "TR02893", "TR02899", "TR02911", "TR02917", "TR02923", "TR02935",
    "TR02947", "TR02959", "TR02971", "TR02983", "TR02989", "TR02995",
    "TR03007", "TR03025", "TR03031", "TR03037", "TR03049", "TR03061",
    "TR03067", "TR03079", "TR03085", "TR03091", "TR03097", "TR03103",
    "TR03109", "TR03115", "TR03121", "TR03127", "TR03139", "TR03145",
    "TR03169", "TR03181", "TR03187", "TR03204", "TR03209", "TR03214",
])


def probe_clip_frames(path, t0=None, t1=None, n=PROBE_FRAMES,
                      span=PROBE_SPAN_SEC):
    """n frames spanning `span` seconds, centred on the labelled event.

    Anomaly clips carry start/end times, so we centre on the part that is
    actually anomalous instead of averaging it away with surrounding normal
    footage. Normal clips have no timings and use the middle of the clip.
    Clips shorter than `span` just use everything they have.
    """
    dur = video_duration(path) or 0.0
    if dur <= 0:
        return []
    if t0 is not None and t1 is not None and np.isfinite(t0) and np.isfinite(t1):
        centre = (float(t0) + float(t1)) / 2
    else:
        centre = dur / 2
    half = min(span, dur) / 2
    lo = max(0.0, min(centre - half, dur - min(span, dur)))
    hi = min(dur, lo + min(span, dur))
    want = np.linspace(lo, max(lo, hi - 1e-3), n)

    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    out = []
    for t in want:
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, int(t * fps)))
        ok, frame = cap.read()
        if not ok:
            continue
        h, w = frame.shape[:2]
        if max(h, w) > CFG.max_side:
            s = CFG.max_side / max(h, w)
            frame = cv2.resize(frame, (int(w * s), int(h * s)))
        out.append(to_pil(frame))
    cap.release()
    return out


def probe_training_rows() -> list[dict]:
    """One row per training clip: path, class, and where the anomaly is.

    Capped per class. The cap is not only about time - normal has 973 clips
    against fire's 77, and an unbalanced fit would learn to say "normal".
    """
    if GT_TRAIN.empty:
        return []
    rows = []
    gt = GT_TRAIN[~GT_TRAIN["video_id"].astype(str).isin(PROBE_DISPUTED_IDS)]
    n_dropped = len(GT_TRAIN) - len(gt)
    if n_dropped:
        print(f"  excluding {n_dropped} clips with disputed labels "
              f"(see PROBE_DISPUTED_IDS)")
    for cls, grp in gt.groupby("class_name"):
        grp = grp.dropna(subset=["path"])
        if len(grp) > PROBE_MAX_PER_CLASS:
            grp = grp.sample(PROBE_MAX_PER_CLASS, random_state=0)
        for _, r in grp.iterrows():
            rows.append({"video_id": r["video_id"], "path": r["path"],
                         "class_name": cls,
                         "t0": r.get("start_time_sec"),
                         "t1": r.get("end_time_sec")})
    return rows


def find_probe_cache():
    """WORK first, then anywhere under /kaggle/input.

    /kaggle/working does not survive a fresh session, so the cache gets
    re-attached as a Kaggle Dataset or Model and found here instead of costing
    another 40-minute embed. rglob covers both - a Dataset mounts at
    /kaggle/input/<slug>/ and a Model at
    /kaggle/input/models/<owner>/<slug>/other/default/1/ - so it does not matter
    which one it was uploaded as.

    The exact name is tried first. Failing that, any train_emb_*.npz is accepted
    with a warning, because Kaggle sometimes renames on upload and a silent
    40-minute re-embed is a worse outcome than a loud approximate match. Shape
    and classes are validated by the caller either way.
    """
    local = WORK / PROBE_CACHE_NAME
    if local.exists():
        return local
    base = Path("/kaggle/input")
    if not base.exists():
        return None
    for p in base.rglob(PROBE_CACHE_NAME):
        return p
    for p in sorted(base.rglob("train_emb_*.npz")):
        print(f"  ! no {PROBE_CACHE_NAME} attached; using {p.name} instead.")
        print(f"    Check it was built at {PROBE_FRAMES} frames over "
              f"{PROBE_SPAN_SEC:.0f}s - a cache from different settings will "
              f"fit and predict happily while meaning something else.")
        return p
    return None


def build_probe_embeddings(force: bool = False):
    """Embed every (capped) training clip once and cache the result.

    The expensive step is turning pixels into vectors; the step we actually want
    to iterate on is fitting a classifier over them. Separating the two turns
    one experiment per twenty minutes into twenty experiments per minute.
    """
    cached = None if force else find_probe_cache()
    if cached is not None:
        z = np.load(cached, allow_pickle=True)
        X, y = z["X"], z["y"]
        print(f"probe cache: {cached}")
        print(f"  {len(y)} clips, {X.shape[1]}-d embeddings, "
              f"{len(set(y.tolist()))} classes")
        # Validate rather than trust. A cache is a file someone attached, and
        # the failure modes are all silent: a truncated upload, a cache built at
        # different settings, or labels outside the twelve would each fit and
        # predict happily while meaning something else.
        ok = True
        if X.shape[0] != len(y):
            print(f"  ! X has {X.shape[0]} rows against {len(y)} labels"); ok = False
        if not np.isfinite(X).all():
            print("  ! cache contains non-finite values"); ok = False
        unknown = set(map(str, y.tolist())) - set(CLASSES)
        if unknown:
            print(f"  ! labels outside the twelve: {sorted(unknown)}"); ok = False
        if len(y) < 200:
            print(f"  ! only {len(y)} clips - too few to fit a 12-class probe"); ok = False
        if not ok:
            print("  ignoring this cache and rebuilding")
        else:
            return X, y, list(z["ids"])

    rows = probe_training_rows()
    if not rows:
        print("no training ground truth - probe unavailable")
        return None, None, None
    print(f"embedding {len(rows)} training clips at {PROBE_FRAMES} frames over "
          f"{PROBE_SPAN_SEC:.0f}s (one-off, cached to {WORK / PROBE_CACHE_NAME})")

    X, y, ids = [], [], []
    t0 = time.time()
    for i, r in enumerate(rows, 1):
        if i % 200 == 0 or i == len(rows):
            el = time.time() - t0
            print(f"  {i}/{len(rows)}  {el/60:.1f} min elapsed, "
                  f"~{el/i*(len(rows)-i)/60:.1f} min left")
        try:
            frames = probe_clip_frames(r["path"], r["t0"], r["t1"])
        except Exception as e:
            print(f"  ! {r['video_id']}: {str(e).splitlines()[0][:90]}")
            continue
        if not frames:
            continue
        # mean-pool the clip's frames into one vector: the probe's unit is a
        # clip, which is what makes persistence classes representable at all
        emb = embed_images(frames).mean(0)
        X.append(emb.detach().float().cpu().numpy())
        y.append(r["class_name"])
        ids.append(r["video_id"])

    X = np.stack(X) if X else np.zeros((0, 768), dtype=np.float32)
    y = np.array(y)
    np.savez_compressed(WORK / PROBE_CACHE_NAME, X=X, y=y, ids=np.array(ids))
    print(f"  wrote {WORK / PROBE_CACHE_NAME}  "
          f"({X.nbytes / 1e6:.1f} MB, {time.time() - t0:.0f}s)")
    return X, y, ids


PROBE_X, PROBE_Y, PROBE_IDS = build_probe_embeddings()

# Apply the exclusion here too, not only at embed time. The cache predates this
# decision and holds all 164 wrong_way clips, and re-embedding to drop 108 rows
# would cost 40 minutes to achieve what one mask does - so filter whatever came
# back, from cache or from a fresh pass.
if PROBE_X is not None and len(PROBE_X):
    _keep = np.array([str(v) not in PROBE_DISPUTED_IDS for v in PROBE_IDS])
    if not _keep.all():
        print(f"  dropping {int((~_keep).sum())} disputed clips from the cache "
              f"({int(_keep.sum())} remain)")
        PROBE_X = PROBE_X[_keep]
        PROBE_Y = PROBE_Y[_keep]
        PROBE_IDS = [v for v, k in zip(PROBE_IDS, _keep) if k]


def fit_probe(X, y, C: float | None = None):
    """Multinomial logistic regression over frozen embeddings.

    Balanced class weights because the cap does not fully level things (fire has
    77 clips against normal's 300), and an unbalanced fit on this data learns
    the majority answer - which is exactly the failure we are trying to remove.

    Reported on a held-out split, not on the training data, because a probe that
    memorises 3,000 embeddings would look excellent and predict nothing.
    """
    C = float(getattr(CFG, "probe_C", 100.0)) if C is None else C
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25,
                                          random_state=0, stratify=y)
    # No multi_class= argument: it was deprecated in sklearn 1.5 and REMOVED in
    # 1.9, where passing it is a TypeError rather than a warning. Multinomial is
    # the default for multiclass with lbfgs, so dropping it changes nothing and
    # works on both old and new versions - Kaggle's image moves without asking.
    clf = LogisticRegression(max_iter=3000, C=C, class_weight="balanced")
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)

    print(f"\nprobe: {len(Xtr)} train / {len(Xte)} held out, "
          f"{len(clf.classes_)} classes")
    print(classification_report(yte, pred, zero_division=0, digits=3))

    print("confusion (rows = truth, cols = predicted), held out:")
    labels = list(clf.classes_)
    cm = confusion_matrix(yte, pred, labels=labels)
    short = [c[:14] for c in labels]
    print("      " + "".join(f"{s:>6}" for s in short))
    for name, row in zip(short, cm):
        print(f"{name:>14}" + "".join(f"{v:>6d}" for v in row))

    # The number that matters for us specifically: can it tell anomalous from
    # normal at all? Class confusion is survivable, saying "normal" is not.
    anom_t = np.array([t != "normal" for t in yte])
    anom_p = np.array([p != "normal" for p in pred])
    tp = int((anom_t & anom_p).sum()); fn = int((anom_t & ~anom_p).sum())
    fp = int((~anom_t & anom_p).sum()); tn = int((~anom_t & ~anom_p).sum())
    print(f"\nanomalous vs normal (the decision stage 1 actually makes):")
    print(f"  recall {tp / max(tp + fn, 1):.3f}   "
          f"precision {tp / max(tp + fp, 1):.3f}   "
          f"TP={tp} FN={fn} FP={fp} TN={tn}")
    print(f"  for comparison, the written health score is INVERTED on 3 of the "
          f"4 test videos measured")
    return clf


PROBE = fit_probe(PROBE_X, PROBE_Y) if PROBE_X is not None and len(PROBE_X) else None


def probe_predict(emb) -> dict:
    """{class_name: probability} for one window's frames.

    Takes the same mean-pooled shape the probe was trained on, so callers must
    pass frames spanning PROBE_SPAN_SEC - handing it a 2s window would be the
    train/test mismatch this cell exists to avoid.
    """
    if PROBE is None:
        return {}
    v = emb.mean(0) if hasattr(emb, "mean") and getattr(emb, "ndim", 1) > 1 else emb
    v = v.detach().float().cpu().numpy() if hasattr(v, "detach") else np.asarray(v)
    p = PROBE.predict_proba(v.reshape(1, -1))[0]
    # str() on the keys: sklearn hands back numpy.str_, which subclasses str and
    # so passes every type check, then serialises into JSON as an object rather
    # than a string. Cast at the boundary instead of debugging it in the arena
    # submission file.
    return {str(c): float(v) for c, v in zip(PROBE.classes_, p.tolist())}


def probe_anomaly(emb) -> float:
    """P(anything is wrong) in [0, 1]. Higher is worse - note this is the
    OPPOSITE sign to health(), which is higher-is-better. Kept explicit rather
    than mimicking the health convention, because silently reusing that sign is
    how the inverted signal went unnoticed for so long."""
    p = probe_predict(emb)
    return float(1.0 - p.get("normal", 1.0)) if p else 0.0


def probe_shortlist(emb, k: int = 5) -> list[str]:
    """The k most likely ANOMALY classes, learned rather than written.

    Held out, the correct class is in the top 5 for 98.3% of anomalous clips
    (top-3: 88.4%). The written shortlist this replaces managed 34% at k=3
    against a 27% baseline for drawing three of eleven at random.
    """
    p = probe_predict(emb)
    if not p:
        return []
    ranked = sorted(((c, v) for c, v in p.items() if c != "normal"),
                    key=lambda kv: -kv[1])
    return [c for c, _ in ranked[:k]]


def probe_window(kept: list[dict], centre_t: float, cfg=None) -> dict:
    """Score the probe on PROBE_SPAN_SEC of video centred on centre_t.

    Costs no GPU. Stage 1 already encoded every kept frame and cell 5 now keeps
    those vectors on the records, so this is a mean over arrays that exist -
    which is also exactly the shape the probe was trained on (8 frames spanning
    16s, mean-pooled), rather than the ~2s window stage 2 uses.

    Returns {} when the probe is unavailable or nothing was sampled nearby, so
    every caller can treat an empty dict as "no opinion".
    """
    cfg = cfg or CFG
    if PROBE is None or not kept:
        return {}
    half = PROBE_SPAN_SEC / 2
    near = [k for k in kept
            if centre_t - half <= k["t"] <= centre_t + half and "emb" in k]
    if not near:
        return {}
    # take PROBE_FRAMES evenly across the span, matching how a training example
    # was built - not the first 8, which would bias to the start of the window
    if len(near) > PROBE_FRAMES:
        idx = np.linspace(0, len(near) - 1, PROBE_FRAMES).round().astype(int)
        near = [near[i] for i in sorted(set(idx.tolist()))]
    v = np.mean([k["emb"] for k in near], axis=0)
    return probe_predict(v)


def probe_curve(kept: list[dict], step_sec: float = 4.0, cfg=None) -> list[tuple]:
    """[(t, P(anomalous)), ...] across a whole video.

    The replacement for the health curve wherever a per-video signal is needed.
    Sampled every step_sec rather than per frame because the probe's unit is a
    16s span - scoring it at 2fps would return sixteen near-identical values.
    """
    cfg = cfg or CFG
    if PROBE is None or not kept:
        return []
    t0, t1 = kept[0]["t"], kept[-1]["t"]
    out = []
    t = t0
    while t <= t1:
        p = probe_window(kept, t, cfg)
        if p:
            out.append((round(float(t), 2), round(1.0 - p.get("normal", 1.0), 4)))
        t += step_sec
    return out


## 5c — Does a purpose-built model read it differently? *(diagnostic, off by default)*

On T025 the pipeline now has **perfect coverage** — all six real events get a VLM
window inside them — every one of eleven classes on offer, an explicit
cause-before-effect instruction, and frames at native resolution cropped to the
motion region. Under all of that, Qwen3-VL-4B writes the same sentence every
time: *"a dense queue of vehicles is stopped or moving very slowly"* →
`traffic_congestion`. Truth says `traffic_accident`, six times.

Coverage, frame filtering, duration and resolution are all eliminated by
measurement. What is left is the model's reading. So ask a second,
independently-trained one: **Cosmos-Embed1**, 1B params LoRA-tuned on
VAD-Reasoning (1,755 videos, 24 anomaly categories), which has a text tower and
can therefore be asked **zero-shot** — no probe to fit, no re-embed.

Both answers are useful. If Cosmos names these correctly, building it in is
clearly worth the time. If it agrees with Qwen, two independently-trained models
disagree with the label and the thing to fix is elsewhere.

Needs `trust_remote_code` (so internet on), and frees the VLM first — three
models will not fit a 16GB T4.

In [ ]:
# =============================================================================
# 5c - Does a purpose-built anomaly model read T025 differently? (DIAGNOSTIC)
# =============================================================================
# Set COSMOS_ENABLED = True to run this. It is off by default because it is a
# question, not a stage: nothing downstream consumes it yet.
#
# THE QUESTION. On T025 the pipeline now has perfect coverage - all six real
# events get a VLM window inside them - every one of eleven classes on offer, an
# explicit instruction to prefer a cause over its effect, and frames delivered
# at native resolution cropped to the motion region. Under all of that,
# Qwen3-VL-4B writes the same sentence every time:
#
#     "A dense queue of vehicles is stopped or moving very slowly on the left
#      side of the highway."   -> traffic_congestion
#
# Ground truth says traffic_accident, six times. We have eliminated coverage
# (6/6 events windowed), frame filtering (density inside events matches
# outside), duration (2s and 16s give identical text) and resolution (640px and
# native give identical text). What remains is the model's reading of the
# footage.
#
# So ask a second, independently-trained model the same question. Cosmos-Embed1
# is 1B params LoRA-tuned on VAD-Reasoning - 1,755 videos across 24 anomaly
# categories - and it has a text tower, so it can be asked zero-shot with no
# probe to fit and no 40-minute re-embed. Two outcomes, both worth having:
#
#   Cosmos says traffic_accident  -> Qwen's reading is the problem, and building
#                                    Cosmos in properly is clearly worth 2 hours
#   Cosmos says congestion too    -> two independently-trained models agree
#                                    against the label, which changes what we
#                                    should be trying to fix
#
# Leaderboard context: entrant #23 scored 40.4 running this model bare, above
# our 37.5, and #16 got 47.1 with a LoRA on top.

COSMOS_ENABLED = False
COSMOS_ID = "nvidia/Cosmos-Embed1-448p-anomaly-detection"
COSMOS_FRAMES = 8          # the shape it was trained at
COSMOS_DIAG_VIDEOS = ["T025", "T032", "T026"]


def load_cosmos(model_id: str = COSMOS_ID):
    """Load Cosmos-Embed1, picking a dtype the GPU can actually run.

    The model card says bfloat16. bf16 needs sm_80 (Ampere) and Kaggle's T4 is
    sm_75, where it is emulated rather than native - so try fp16 first, which
    the T4 does have hardware for, and fall back to fp32. Getting this wrong is
    not an error message, it is a silently slow run.

    trust_remote_code=True is required: the architecture ships as custom code on
    the Hub rather than living in transformers, so the notebook needs internet
    enabled. That is a real precondition, not a detail.
    """
    from transformers import AutoModel, AutoProcessor
    last = None
    for dt in (torch.float16, torch.float32):
        try:
            m = AutoModel.from_pretrained(model_id, trust_remote_code=True,
                                          torch_dtype=dt).to(DEVICE).eval()
            p = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
            print(f"cosmos: {model_id} loaded as {dt}")
            return m, p, dt
        except Exception as e:
            last = e
            print(f"  {dt} failed: {str(e).splitlines()[0][:140]}")
    raise RuntimeError(f"could not load {model_id}: {last}")


@torch.no_grad()
def cosmos_video_embedding(frames_bgr: list, model, proc, dtype):
    """One 768-d L2-normalised embedding for a clip of BGR frames.

    Input to the processor is (B, T, C, H, W) with values in 0..1 - the model
    card's example transposes (1, T, H, W, 3) by (0, 1, 4, 2, 3) to get there.
    The processor handles the resize to 448.
    """
    if not frames_bgr:
        return None
    idx = np.linspace(0, len(frames_bgr) - 1, COSMOS_FRAMES).round().astype(int)
    picked = [frames_bgr[i] for i in idx]
    rgb = np.stack([cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in picked])  # T,H,W,3
    batch = np.transpose(rgb[None, ...], (0, 1, 4, 2, 3))                 # 1,T,3,H,W
    inputs = proc(videos=batch)
    inputs = {k: (v.to(DEVICE, dtype=dtype) if hasattr(v, "to") else v)
              for k, v in inputs.items()}
    out = model.get_video_embeddings(**inputs)
    v = getattr(out, "visual_proj", out)
    return v.float().cpu().numpy().reshape(-1)


@torch.no_grad()
def cosmos_class_embeddings(model, proc, dtype):
    """Text embeddings for the eleven anomaly classes plus normal.

    Phrased as short scene descriptions rather than bare label strings, because
    the model was trained against captions - "a traffic accident with collided
    vehicles" sits closer to its training distribution than
    "traffic_accident" does.
    """
    prompts = {
        "normal": "ordinary traffic flowing normally with nothing unusual",
        "traffic_accident": "a traffic accident, vehicles collided or overturned",
        "traffic_congestion": "heavy traffic congestion, a long queue of slow vehicles",
        "stalled_or_broken_down_vehicle": "a broken down vehicle stopped at the roadside",
        "vehicle_blocking_traffic": "a vehicle blocking the road so others cannot pass",
        "wrong_way_driving": "a vehicle driving the wrong way against the traffic",
        "road_spill_or_debris": "spilled cargo or debris scattered on the road surface",
        "waterlogging_or_flood": "a flooded road covered in standing water",
        "fire": "flames and fire burning",
        "smoke": "thick smoke rising",
        "fighting_or_violence": "people fighting, punching or brawling",
        "loitering_or_suspicious_presence": "a person loitering, standing around suspiciously",
    }
    names = list(prompts)
    inputs = proc(text=[prompts[n] for n in names])
    inputs = {k: (v.to(DEVICE) if hasattr(v, "to") else v) for k, v in inputs.items()}
    out = model.get_text_embeddings(**inputs)
    t = getattr(out, "text_proj", out)
    return names, t.float().cpu().numpy()


def cosmos_zeroshot(frames_bgr: list, model, proc, dtype, names, T):
    """{class: probability} by video-text similarity. No probe, no training."""
    v = cosmos_video_embedding(frames_bgr, model, proc, dtype)
    if v is None:
        return {}
    sims = T @ v
    e = np.exp((sims - sims.max()) * 100.0)      # 100 ~ the model's logit scale
    return dict(zip(names, (e / e.sum()).tolist()))


if not COSMOS_ENABLED:
    print("cell 5c: COSMOS_ENABLED is False - set it True to run the diagnostic")
else:
    # Free the VLM first if it is resident. Qwen-4B is ~9.5GB and SigLIP2 ~1.5GB;
    # adding a 1B model on a 16GB T4 is close enough to the edge that the answer
    # would be an OOM rather than a verdict. This cell is a diagnostic, so it may
    # cost a Qwen reload afterwards - cell 6 will notice and reload on its own.
    if "vlm" in globals():
        print(f"cosmos: freeing the VLM first "
              f"({free_cuda('vlm', 'vlm_proc', '_VLM_ID'):.2f} GB in use)")

    _cm, _cp, _cd = load_cosmos()
    _names, _T = cosmos_class_embeddings(_cm, _cp, _cd)
    print(f"cosmos: {len(_names)} class prompts embedded, "
          f"{_T.shape[1]}-d\n")

    for _vid in COSMOS_DIAG_VIDEOS:
        _path = VIDEO_PATHS.get(_vid)
        if _path is None:
            print(f"  ! {_vid} not found")
            continue
        _truth = GT_TEST[(GT_TEST.video_id == _vid)
                         & GT_TEST.start_time_sec.notna()]
        if _truth.empty:
            continue
        print(f"{_vid}  truth: {sorted(set(_truth.class_name))}")
        _kept, _ = sample_video(_path, CFG)
        for _, _t in _truth.iterrows():
            # the frames inside this real event, which is the fair test - we
            # already know coverage is not the problem here
            _win = [k["frame"] for k in _kept
                    if _t.start_time_sec <= k["t"] <= _t.end_time_sec]
            if len(_win) < 2:
                print(f"   {_t.start_time_sec:6.0f}-{_t.end_time_sec:<6.0f} "
                      f"only {len(_win)} frames, skipped")
                continue
            _p = cosmos_zeroshot(_win, _cm, _cp, _cd, _names, _T)
            _top = sorted(_p.items(), key=lambda kv: -kv[1])[:3]
            _hit = "CORRECT" if _top[0][0] == _t.class_name else ""
            print(f"   {_t.start_time_sec:6.0f}-{_t.end_time_sec:<6.0f} "
                  + "  ".join(f"{c[:22]}:{v:.2f}" for c, v in _top) + f"   {_hit}")
        print()
    print("If Cosmos names these correctly, building it in is worth the time.")
    print("If it agrees with Qwen, two independent models disagree with the label")
    print("and the thing to fix is somewhere else entirely.")


## 6 — Stage 2: the small VLM

Runs only on what stage 1 could not clear. **Qwen3-VL-4B**, not the 3B this
started as — that was sized for a 4GB laptop GPU and left most of a T4's 16GB
unused. Qwen3-VL adds video-specific architecture (interleaved MRoPE, textual
timestamps, temporally dense captions) Qwen2.5-VL lacks. It's also independently
validated for this exact task: QVAD (arXiv:2604.03040), a VAD paper in the
organizers' own SOTA deck, uses Qwen3-VL-4B-Instruct for captioning, and 2 of
the top 3 accepted-paper teams on the AI City Challenge 2026 traffic-anomaly
leaderboard ran Qwen3-VL-8B.

Two things keep it cheap and honest beyond the model swap:

**Shortlisting** — stage 1's embedding already ranks the eleven anomaly labels,
so we ask about the top three. Shorter prompt, and the model is not invited to
hallucinate through eight irrelevant options.

**ASK-Hint prompting** — every label expands into concrete, observable
questions. "Is there any anomaly?" misses what "Do you see punching, kicking, or
wrestling on the ground?" catches on the same input. This is a text file rather
than a training run, which makes it the best accuracy-per-minute available.

In [ ]:
# =============================================================================
# 5 - Stage 2: small VLM verification on escalated windows only
# =============================================================================
# Runs on the ~12% of frames stage 1 could not clear. Two things make this
# cheaper and more accurate than "show the VLM a frame and ask if it is weird":
#
# 1. SHORTLISTING. Stage 1's frame embedding already ranks the twelve labels.
#    We only ask about the top few, so the prompt stays short and the model is
#    not invited to hallucinate its way through nine irrelevant options.
#
# 2. ASK-HINT PROMPTING (WACV 2026). Abstract prompts fail where action-centric
#    ones succeed - "Is there any anomaly?" misses what "Do you see punching,
#    kicking, or wrestling on the ground?" catches on the same input. So every
#    label expands into concrete, observable questions rather than being handed
#    to the model as a bare class string. This is a text file, not a training
#    run: the highest accuracy-per-minute available today.

import subprocess
import sys

from transformers import AutoProcessor as VLMProcessor

# Descriptions used for the stage-1 shortlist (embedding space, not the VLM).
CLASS_DESCRIPTIONS = {
    "traffic_accident": [
        "a car crash with damaged vehicles on the road",
        "two vehicles collided at an intersection",
        "an overturned vehicle on its side after a crash",
    ],
    "traffic_congestion": [
        "a long queue of stationary vehicles filling the road",
        "heavy traffic jam with cars bumper to bumper",
    ],
    "stalled_or_broken_down_vehicle": [
        "a single vehicle stopped on the hard shoulder with hazard lights",
        "a broken down car stationary in a live traffic lane",
    ],
    "vehicle_blocking_traffic": [
        "a vehicle parked across the road obstructing other cars",
        "a truck blocking a junction so traffic cannot pass",
    ],
    "wrong_way_driving": [
        "a vehicle driving towards oncoming traffic",
        "a car travelling the wrong way down a one way road",
    ],
    "road_spill_or_debris": [
        "debris and scattered objects lying across the road surface",
        "a spilled load of cargo covering the carriageway",
    ],
    "waterlogging_or_flood": [
        "a road submerged under standing flood water",
        "vehicles driving through deep water on a flooded street",
    ],
    "fire": [
        "an open flame burning on a vehicle or building",
        "a fire with visible orange flames in the scene",
    ],
    "smoke": [
        "thick smoke rising and spreading across the scene",
        "a plume of grey smoke obscuring the view",
    ],
    "fighting_or_violence": [
        "two people physically fighting and throwing punches",
        "a violent altercation between people in the street",
    ],
    "loitering_or_suspicious_presence": [
        "a person lingering in a restricted area for a long time",
        "someone loitering near parked vehicles at night",
    ],
}

# ASK-Hint question banks. Concrete and observable - each one should be
# answerable by looking, without inference about intent.
ASK_HINT = {
    "traffic_accident": [
        "Do you see two or more vehicles in contact, or a vehicle that has struck something?",
        "Is any vehicle visibly damaged, overturned, or off its wheels?",
        "Are people gathered around a stopped vehicle in the roadway?",
    ],
    "traffic_congestion": [
        "Is there a dense queue of vehicles that are stopped or barely moving?",
        "Does the queue extend across most of the visible road?",
    ],
    "stalled_or_broken_down_vehicle": [
        "Is a single vehicle stationary while other traffic moves past it?",
        "Is it stopped on a shoulder, in a live lane, or somewhere vehicles do not normally park?",
        "Are hazard lights on, a bonnet open, or a warning triangle placed?",
    ],
    "vehicle_blocking_traffic": [
        "Is a vehicle positioned so that other vehicles cannot get past?",
        "Is a vehicle stopped across a junction, crossing, or lane?",
    ],
    "wrong_way_driving": [
        "Is any vehicle facing or moving opposite to the other vehicles around it?",
        "Is a vehicle on the wrong side of a divided road or driving against arrows and markings?",
    ],
    "road_spill_or_debris": [
        "Are there objects, rubble, cargo, or scattered material on the road surface?",
        "Are vehicles swerving or slowing to avoid something lying on the road?",
    ],
    "waterlogging_or_flood": [
        "Is part of the road covered by standing water?",
        "Are vehicle wheels partly submerged, or is water rippling across the surface?",
    ],
    "fire": [
        "Do you see open flames anywhere in the scene?",
        "Is a vehicle, building, or pile of material actively burning?",
    ],
    "smoke": [
        "Do you see smoke rising or drifting across the scene?",
        "Is visibility reduced by a plume of smoke rather than by fog or rain?",
    ],
    "fighting_or_violence": [
        "Do you see punching, kicking, grappling, or pushing between people?",
        "Is anyone on the ground while others stand over them?",
        "Is a crowd reacting to or surrounding a physical confrontation?",
    ],
    "loitering_or_suspicious_presence": [
        "Is a person remaining in one place for an unusually long time?",
        "Is someone lingering near vehicles, doors, or fences without an obvious purpose?",
        "Is a person in an area that is otherwise empty of people?",
    ],
}

CLASS_EMB_TEXTS, CLASS_EMB_OWNER = [], []
for cls, descs in CLASS_DESCRIPTIONS.items():
    CLASS_EMB_TEXTS += descs
    CLASS_EMB_OWNER += [cls] * len(descs)
CLASS_EMB = embed_texts(CLASS_EMB_TEXTS)
CLASS_OWNER = np.array(CLASS_EMB_OWNER)


def shortlist_classes(img_emb: torch.Tensor, k: int = 5) -> list[str]:
    """Rank the eleven anomaly labels for a window by max similarity.

    Note this is NOT used as a detector - Alert-CLIP shows CLIP-family text
    embeddings for normal vs abnormal are entangled enough that raw similarity
    is a poor yes/no. It is used only to decide which questions to ask, where
    being roughly right is sufficient and being wrong just wastes a question.

    That last sentence was false for the whole first run, and it cost us most of
    our score. See build_prompt() below: the shortlist was also injected into
    the required JSON schema, so being wrong did not waste a question - it
    deleted the correct answer. The shortlist is now a hint and nothing else,
    which is what this docstring always claimed.

    k is 5 rather than 3 because the shortlist no longer restricts anything, so
    a wider hint costs only prompt length. It was measured at 34% hit rate for
    k=3 against a 27% random baseline, i.e. very nearly uninformative; widening
    it is a stopgap until the linear probe replaces this ranking entirely.
    """
    # Prefer the learned ranking when cell 5b produced one. Held out, the correct
    # class is in the probe's top 5 for 98.3% of anomalous clips against 34% for
    # this text-similarity ranking at k=3 - which is barely above the 27% you get
    # by drawing three of eleven at random. The text version stays as the
    # fallback for a run where the probe could not be fitted.
    if "probe_shortlist" in globals() and PROBE is not None:
        learned = probe_shortlist(img_emb, k=k)
        if learned:
            return learned
    sim = (img_emb.mean(0, keepdim=True) @ CLASS_EMB.T).squeeze(0)
    best = {}
    for s, owner in zip(sim.tolist(), CLASS_OWNER):
        best[owner] = max(best.get(owner, -9.9), s)
    return [c for c, _ in sorted(best.items(), key=lambda kv: -kv[1])[:k]]


# --- load the VLM -------------------------------------------------------------
# sdpa, not flash-attention-2: FA2 needs sm_80+ and Kaggle's T4 is sm_75. Asking
# for it fails at load, not at generate, which is at least an honest error.
def load_vlm(model_id=None):
    """Qwen3-VL-4B, not Qwen2.5-VL-3B and not Qwen3-VL-8B.

    3B was a compromise for the GTX 1650's 4GB VRAM ceiling - irrelevant on a
    T4 (16GB). Measured VRAM: Qwen3-VL-4B ~9-10GB fp16 (comfortable alongside
    SigLIP2's ~1.5GB), Qwen3-VL-8B ~19GB fp16 / ~12GB in 4-bit ("on the edge"
    per multiple sources - not worth the OOM risk on a live run). Qwen3-VL adds
    video-specific architecture (interleaved MRoPE, textual timestamps,
    temporally dense captions) that Qwen2.5-VL lacks, and beats Qwen2.5-VL-7B on
    11/12 shared benchmarks including the video ones (CharadesSTA, LVBench).

    Independently validated for THIS exact task: QVAD (arXiv:2604.03040), a
    training-free VAD paper in the organizers' own SOTA deck, uses
    Qwen3-VL-4B-Instruct for captioning. AI City Challenge 2026 Track 3
    (traffic anomalies) had 2 of the top 3 accepted-paper teams on Qwen3-VL-8B.

    Needs transformers>=4.57.0 (Qwen3-VL shipped Oct 2025); cell 5 may have
    already imported an older version, so upgrade defensively and fall back to
    Qwen2.5-VL-3B (known-good) rather than leave the notebook dead mid-session.
    """
    model_id = model_id or CFG.vlm_id
    try:
        from transformers import Qwen3VLForConditionalGeneration as VLMClass
    except ImportError:
        print("transformers too old for Qwen3-VL - upgrading (needs >=4.57.0)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                        "transformers>=4.57.0"], check=False)
        try:
            from transformers import Qwen3VLForConditionalGeneration as VLMClass
        except ImportError as e:
            print(f"still unavailable after upgrade ({e}); "
                  "falling back to Qwen2.5-VL-3B-Instruct")
            model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
            from transformers import Qwen2_5_VLForConditionalGeneration as VLMClass

    proc = VLMProcessor.from_pretrained(model_id)
    # Cap the vision token count. This family is resolution-native, so an
    # uncapped 640px frame can cost >1500 tokens per image; at 4 images per
    # window that alone decides whether this fits on a T4 and whether it is
    # 3 fps or 0.5 fps.
    if hasattr(proc, "image_processor"):
        proc.image_processor.min_pixels = 256 * 28 * 28
        proc.image_processor.max_pixels = 768 * 28 * 28
    m = VLMClass.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if USE_FP16 else torch.float32,
        attn_implementation="sdpa",
        device_map="auto" if DEVICE == "cuda" else None,
        low_cpu_mem_usage=True,
    ).eval()
    return m, proc


# The VLM is ~9-10GB in fp16 and the T4 has 16, so a second copy does not fit.
# Re-running this cell rebinds `vlm` but the old module stays referenced until
# the new one is built, which means two copies momentarily resident and a CUDA
# out-of-memory error rather than a slow reload. Reuse when it is already here,
# and free before loading when it is not.
if globals().get("_VLM_ID") == CFG.vlm_id and "vlm" in globals():
    print(f"vlm: {CFG.vlm_id} already loaded, reusing")
else:
    if "_VLM_ID" in globals():
        _free = free_cuda("vlm", "vlm_proc")
        print(f"vlm: freed the previous model ({_free:.2f} GB still in use)")
    vlm, vlm_proc = load_vlm()
    _VLM_ID = CFG.vlm_id
    print(f"vlm: {CFG.vlm_id} loaded on {DEVICE}")
if DEVICE == "cuda":
    print(f"     VRAM in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


SYSTEM = (
    "You are a video surveillance analyst reviewing a few consecutive frames from "
    "one camera. Answer only from what is visible. If the scene looks like ordinary "
    "activity, say so - false alarms are as costly as missed events."
)


# A queue of stopped cars is what an accident LOOKS like from above once the
# first second is over, and the label is the accident. Measured on T025, where
# the truth is six traffic_accident events and, given all eleven classes and no
# constraint, the model wrote:
#
#   "A dense queue of vehicles is stopped or moving very slowly on the highway."
#   "A long queue of trucks and cars is stationary at the service area entrance,
#    indicating traffic congestion."
#
# Those are accurate descriptions and the wrong answer. The ASK-HINT questions
# for traffic_accident are good ones - vehicles in contact, visible damage,
# people gathered - and the model answered them honestly with "no", then picked
# the class whose questions it could answer "yes" to. Nothing about that is a
# failure of prompting the individual classes; what was missing is an ordering
# between them.
#
# Note what this deliberately is NOT: a lookup table mapping consequence to
# cause. We rejected that earlier and the reasoning still holds - smoke over a
# wrecked car is a symptom, smoke over a thermal plant is the incident, and no
# table can tell those apart. This asks the model to LOOK for a cause before
# settling on an effect, and leaves the judgement where the eyes are.
CAUSE_BEFORE_EFFECT = (
    "One ordering rule. Stopped traffic, a queue, a blocked lane and a crowd are "
    "usually consequences of something else. If you see one, look at the head of "
    "the queue or the centre of the crowd before you answer: a collision, a "
    "damaged or overturned vehicle, debris, water or fire there is the incident, "
    "and the queue is only its effect. Report the cause when you can see one, and "
    "the effect only when you cannot."
)


def build_prompt(hint_classes: list[str]) -> str:
    """Ask about the shortlisted classes; accept an answer from all eleven.

    hint_classes chooses which ASK-HINT question banks to spell out. It does NOT
    restrict what the model may answer - the schema below always offers every
    anomaly class plus "normal".

    That separation is the single highest-value fix in this project, because the
    previous version collapsed it. `candidates` went into the required JSON
    schema, so the model had to reply with one of three classes or "normal".
    Measured on the practice pack, over the 47 windows that actually overlapped
    a real ground-truth event:

        correct class present in the 3-way shortlist   34%   (random 3-of-11: 27%)
        model answered "normal"                        79%
        right class never on the menu at all           31 of 47   (66%)

    So two thirds of our misses were a multiple-choice question with the correct
    answer removed, and "normal" was the only remaining option that was not
    definitionally wrong. Of 26 ground-truth timed events, 23 had a window
    overlapping them and only 3 had a window of the right class - we were
    looking at 88% of real events and recognising 12%.

    The question banks stay shortlisted (top 5, not all 11) so the prompt does
    not quadruple in length and dilute attention across sixty-odd questions.
    """
    lines = [
        "These frames were flagged by an automatic filter. Decide whether they show "
        "a genuine incident that a responder should be sent to.",
        "",
        "These checks are the most likely possibilities, not the only ones - if "
        "what you see is a different kind of incident, name that instead:",
    ]
    for cls in hint_classes:
        lines.append(f"\n[{cls}]")
        lines += [f"  - {q}" for q in ASK_HINT.get(cls, [])]
    lines += [
        "",
        "If a red circle or square is drawn on a frame, it marks where motion was "
        "detected - look there first, but judge the whole frame.",
        "",
        CAUSE_BEFORE_EFFECT,
        "",
        "Reply with JSON only, no other text:",
        '{"anomaly": true|false, "class": "<one of: '
        + ", ".join(ANOMALY_CLASSES + ["normal"]) + '>", '
        '"confidence": <0.0-1.0>, "description": "<one short sentence>"}',
    ]
    return "\n".join(lines)


def resolve_class(raw: str) -> str:
    """Map a model's class string onto one of the twelve, tolerantly.

    Exact-match-or-normal was safe while the schema offered three options the
    model could copy verbatim. Now that it chooses freely from eleven, a reply
    of "traffic accident" or "Traffic_Accident" would be silently scored as
    normal - reintroducing the same failure this change exists to remove, just
    one layer further down. Normalise separators and case before giving up.
    """
    s = str(raw or "").strip().lower().replace("-", "_").replace(" ", "_")
    s = re.sub(r"_+", "_", s).strip("_")
    if s in CLASSES:
        return s
    squashed = {c.replace("_", ""): c for c in CLASSES}
    return squashed.get(s.replace("_", ""), "normal")


@torch.no_grad()
def vlm_pick_class(pil_frames: list, options: list[str]) -> dict | None:
    """Forced choice among `options`. No "normal", because the caller already
    decided something IS happening.

    This exists for exactly one measured failure. On T025 the probe correctly
    localises five of six real accidents - probe-span extents score IoU 0.800
    against the ground truth - and then names every one of them
    wrong_way_driving. Extent right, class wrong, five events lost.

    The division of labour that fixes it: the PROBE decides WHETHER (98% recall,
    from 16s of temporal context), the VLM decides WHICH (it can actually see).
    Asking the probe to do both wastes the VLM, and the probe's top-1 is 0.739
    against 0.884 for its top-3 - so handing the VLM those three and making it
    choose is worth about fifteen points of class accuracy if the VLM can pick
    at all.

    Returns None on any failure, so the caller keeps the probe's own answer.
    """
    if not options:
        return None
    # `options` narrows WHICH QUESTIONS get spelled out. It does NOT narrow the
    # answer - the schema below offers all eleven.
    #
    # The first version of this function did narrow the answer, and it rebuilt
    # the exact bug cell 6 exists to fix, one layer down. Measured on T025:
    # traffic_accident was in the probe's top 3 for 1 window out of 12, so the
    # re-ask could not answer it however clearly the frames showed one. The VLM
    # dutifully picked stalled_or_broken_down_vehicle off the menu it was given,
    # and five events that pass the IoU gate at 0.82-0.98 stayed wrong.
    #
    # A shortlist is a hint about where to look. The moment it reaches the reply
    # schema it stops being a hint and starts deleting correct answers.
    lines = [
        "Something in these frames has been flagged as an incident by an "
        "automatic system, and you should assume it is right about that.",
        "",
        "Your job is to say WHICH incident it is. These are the most likely "
        "candidates, but you may answer with any class in the list at the end:",
    ]
    for c in options:
        lines.append(f"\n[{c}]")
        lines += [f"  - {q}" for q in ASK_HINT.get(c, [])]
    lines += [
        "",
        CAUSE_BEFORE_EFFECT,
        "",
        "Pick the single best fit even if you are unsure. Reply with JSON only:",
        '{"class": "<one of: ' + ", ".join(ANOMALY_CLASSES) + '>", '
        '"confidence": <0.0-1.0>, "description": "<one short sentence>"}',
    ]
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [{"type": "image"} for _ in pil_frames]
         + [{"type": "text", "text": "\n".join(lines)}]},
    ]
    try:
        text = vlm_proc.apply_chat_template(messages, tokenize=False,
                                            add_generation_prompt=True)
        inputs = vlm_proc(text=[text], images=pil_frames, return_tensors="pt",
                          padding=True)
        inputs = {k: (v.to(vlm.device) if hasattr(v, "to") else v)
                  for k, v in inputs.items()}
        out = vlm.generate(**inputs, max_new_tokens=CFG.vlm_max_new_tokens,
                           do_sample=False, temperature=None, top_p=None, top_k=None)
        gen = out[0][inputs["input_ids"].shape[1]:]
        d = parse_json_reply(vlm_proc.decode(gen, skip_special_tokens=True))
    except Exception:
        return None
    # It may still answer "normal" despite not being offered it - that is the
    # model declining the premise, and the caller's probe evidence outranks a
    # refusal to choose, so treat it as no opinion rather than as a veto.
    return d if d.get("class") in ANOMALY_CLASSES else None


def parse_json_reply(text: str) -> dict:
    """Small models wrap JSON in prose or fences often enough that a bare
    json.loads is a reliability bug, not a shortcut."""
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            d = json.loads(m.group(0))
            cls = resolve_class(d.get("class", "normal"))
            conf = float(d.get("confidence", 0.0))
            return {
                "anomaly": bool(d.get("anomaly", False)) and cls != "normal",
                "class": cls,
                "confidence": max(0.0, min(1.0, conf)),
                "description": str(d.get("description", ""))[:300],
                "raw": text,
            }
        except Exception:
            pass
    # Fall back to keyword rescue rather than dropping the window entirely.
    low = text.lower()
    hit = next((c for c in ANOMALY_CLASSES if c.replace("_", " ") in low), None)
    return {"anomaly": hit is not None, "class": hit or "normal",
            "confidence": 0.4 if hit else 0.0, "description": text.strip()[:300],
            "raw": text}


@torch.no_grad()
def vlm_verify(pil_frames: list, candidates: list[str]) -> dict:
    _t0 = time.time()
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [{"type": "image"} for _ in pil_frames]
         + [{"type": "text", "text": build_prompt(candidates)}]},
    ]
    text = vlm_proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = vlm_proc(text=[text], images=pil_frames, return_tensors="pt", padding=True)
    inputs = {k: (v.to(vlm.device) if hasattr(v, "to") else v) for k, v in inputs.items()}
    out = vlm.generate(**inputs, max_new_tokens=CFG.vlm_max_new_tokens,
                       do_sample=False, temperature=None, top_p=None, top_k=None)
    gen = out[0][inputs["input_ids"].shape[1]:]
    result = parse_json_reply(vlm_proc.decode(gen, skip_special_tokens=True))
    # "vision-language-model" matches the PDF's own model_runtimes example name.
    _log_call("vision-language-model", (time.time() - _t0) * 1000)
    return result


ADJUDICATE_SYSTEM = (
    "You are a video surveillance analyst. Several separate observations were made "
    "at the same location within a short time. Decide what single incident best "
    "explains them together, judging only from the frames."
)


@torch.no_grad()
def adjudicate_primary(pil_frames: list, observations: list[dict]) -> dict | None:
    """Given several class verdicts inside one temporal cluster, pick the ONE
    primary incident - by asking the model, not by consulting a causal table.

    Why not a table: a static cause->consequence map has to decide once and for
    all what smoke "means", and smoke is a symptom over a wrecked car but the
    primary event over a thermal plant or a hillside. The same class changes
    role with context, so any fixed tree is wrong in whichever context it did
    not anticipate. The VLM already sees the context, so it is the right thing
    to ask - one extra call per multi-class cluster, a handful per video.

    Returns None on any failure; the caller then falls back to the highest
    confidence observation, so this can only improve on that baseline.
    """
    seen = []
    for o in sorted(observations, key=lambda o: o["t0"]):
        seen.append(f"  - at {o['t0']:.0f}s: {o['class']} ({o['confidence']:.2f}) "
                    f"- {o.get('description', '')[:110]}")
    prompt = "\n".join([
        "These observations were made at one location, in this order:",
        *seen,
        "",
        "They may be several views of ONE incident, or genuinely separate things.",
        "Pick the single class that best describes the primary incident here. If "
        "the observations are consequences of something else visible in the frames "
        "(for example smoke and a gathered crowd around damaged vehicles), name "
        "that underlying incident instead.",
        "",
        "Reply with JSON only:",
        '{"primary": "<one of: ' + ", ".join(ANOMALY_CLASSES) + '>", '
        '"confidence": <0.0-1.0>, "reason": "<one short sentence>"}',
    ])
    try:
        _t0 = time.time()
        messages = [
            {"role": "system", "content": [{"type": "text", "text": ADJUDICATE_SYSTEM}]},
            {"role": "user", "content": [{"type": "image"} for _ in pil_frames]
             + [{"type": "text", "text": prompt}]},
        ]
        text = vlm_proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = vlm_proc(text=[text], images=pil_frames, return_tensors="pt", padding=True)
        inputs = {k: (v.to(vlm.device) if hasattr(v, "to") else v) for k, v in inputs.items()}
        out = vlm.generate(**inputs, max_new_tokens=120, do_sample=False,
                           temperature=None, top_p=None, top_k=None)
        gen = out[0][inputs["input_ids"].shape[1]:]
        raw = vlm_proc.decode(gen, skip_special_tokens=True)
        _log_call("vision-language-model", (time.time() - _t0) * 1000)

        m = re.search(r"\{.*\}", raw, re.S)
        if not m:
            return None
        d = json.loads(m.group(0))
        cls = str(d.get("primary", "")).strip()
        if cls not in ANOMALY_CLASSES:
            return None
        return {"primary": cls,
                "confidence": max(0.0, min(1.0, float(d.get("confidence", 0.5)))),
                "reason": str(d.get("reason", ""))[:200]}
    except Exception as e:
        print(f"  ! adjudication failed: {str(e).splitlines()[0][:100]}")
        return None


# --- smoke test ---------------------------------------------------------------
if _probe is not None:
    _k, _ = stage1_video(_probe)
    if _k:
        _worst = sorted(_k, key=lambda r: r["health"])[:CFG.vlm_frames]
        _worst = sorted(_worst, key=lambda r: r["t"])
        _pil = [to_pil(draw_visual_prompt(r["frame"], r["box"], CFG.visual_prompt)) for r in _worst]
        _emb = embed_images(_pil)
        _cands = shortlist_classes(_emb)
        _t0 = time.time()
        _res = vlm_verify(_pil, _cands)
        print(f"\n{_probe.name}  candidates={_cands}  ({time.time() - _t0:.1f}s)")
        print(json.dumps({k: v for k, v in _res.items() if k != "raw"}, indent=2))


## 7 — Per-class temporal aggregation

The PS's own table says an accident is over in ~1s, congestion builds gradually,
a stalled vehicle is anomalous only after persisting, and waterlogging is a
static condition. That is four different aggregators, not one — a min-duration
long enough to stop congestion flickering erases every accident.

So persistence is per class, and events open/close with **hysteresis**. That is
where false-alarm suppression lives: one confident window opens nothing, while a
real event survives a brief occlusion instead of fragmenting into five alerts.

In [ ]:
# =============================================================================
# 7 - Temporal aggregation: window verdicts -> incidents
# =============================================================================
# Three jobs, in order:
#   1. cluster detections that belong to the same incident (across classes)
#   2. give each cluster ONE main tag plus sub-tags for everything else seen
#   3. give it a real start and end, measured rather than assumed
#
# The problem statement is explicit that events do not share a temporal shape:
# an accident is over in about a second, congestion builds gradually, a stalled
# vehicle is only anomalous AFTER standing a while, waterlogging is a static
# condition. So persistence is per class, and events open/close with hysteresis
# - one confident window opens nothing, a real event survives a brief occlusion
# instead of fragmenting.

# min_dur = how long a detection must persist before it counts as this class.
# MEASURED, not reasoned: the first version came from the PS's prose and was
# badly wrong - loitering was set to 20s when the timed ground truth has a
# MEDIAN of 13.7s, and stalled_vehicle to 15s when the training clips for that
# class average 11.2s in total. "Only anomalous after standing a while" is true
# of the phenomenon and says nothing about how the dataset clipped it.
TEMPORAL = {
    "traffic_accident":                 1.0,   # PS: over in ~1s; GT min 5.0s
    "traffic_congestion":               2.5,   # GT min 5.0s
    "stalled_or_broken_down_vehicle":   4.0,   # train clips avg 11.2s total
    "vehicle_blocking_traffic":         3.0,   # GT 9.5s
    "wrong_way_driving":                1.5,   # short but unambiguous
    "road_spill_or_debris":             2.0,   # static condition
    "waterlogging_or_flood":            2.0,   # static condition
    "fire":                             1.5,
    "smoke":                            2.0,
    "fighting_or_violence":             1.5,   # GT 60s, train clips are short
    "loitering_or_suspicious_presence": 3.0,   # GT min 2.6s - NOT 20s
}
DEFAULT_MIN_DUR = 2.0

# Detections within this many seconds are treated as one incident, REGARDLESS
# of class - the smoke at 507s and the altercation at 523s in T033 are two
# views of one accident, not two incidents. The arena scores only the
# best-overlapping prediction per real event and counts every other one
# against you, so fragmenting is penalised twice over.
# Must tolerate a scan slot or two coming back "normal" in the middle of a real
# event. At a 20s scan interval a single normal slot opens a 40s gap, and 30s
# split T031's fourteen agreeing congestion windows into three separate events -
# each then emitted at the 15s fallback, turning one 125s event into three
# fragments that the arena penalises. 60s = three scan slots.
# --- REMOVED: CROSS_CLASS_GAP_SEC / SAME_CLASS_GAP_SEC ------------------------
# Both were time constants standing in for a question the data already answers.
# T031 settles it: fourteen windows all calling traffic_congestion span 9s-311s
# at a 20s scan stride, and the truth is ONE 125s event at 235-360. Merging all
# fourteen gives IoU 0.22; splitting them all gives ~0.02 each. No value of a
# gap constant produces the right answer, because the right question is not how
# far apart two detections are but whether anything happened in between - and
# the health curve records exactly that. See _recovered_between().

# --- how long was it, really? ------------------------------------------------
# Our window boundaries measure OUR SAMPLING, not the event: 4 frames at 2fps is
# a ~2s window because that is how we look, not because the event lasted 2s.
# Median predicted duration was 4.0s against a real median of 20s, and IoU
# between those is 0.20 - below the arena's 0.5 gate even with a perfect class
# and perfect centring. That made 75 of 100 marks (D2+D3) unreachable no matter
# how good detection got.
#
# So measure the extent instead of assuming it: walk outward from the detection
# while the health score stays depressed. Verified on T033's real predictions -
# the walk returns [507.0, 533.5] against a true event of [490, 535], IoU 0.589,
# a pass. The same detections under a fixed 10s expansion give IoU 0.444, a
# fail. Measurement beats the constant.
EXTENT_LOOSE_FACTOR = 0.4    # walk while health < health_thresh * this.
                             # Kept, because it is a FRACTION OF A CALIBRATED
                             # NUMBER, not a guessed duration - health_thresh is
                             # fitted on known-normal clips, and this says "still
                             # depressed" relative to it.

# --- REMOVED: EXTENT_MAX_SEC (180s cap) --------------------------------------
# It made any event longer than 360s unmatchable, and the evaluation pack's
# E027 is exactly that: one event spanning ~600s of a 602s video. A cap on how
# long reality is allowed to be is not a safeguard, it is an assertion.

# --- REMOVED: FALLBACK_EVENT_SEC (15s floor) ---------------------------------
# The honest reason it existed: when the health curve gave no usable extent, we
# had nothing to say and said "15 seconds" anyway. That is not a measurement,
# and it made 8 of 26 ground-truth events (31%) unmatchable by construction -
# IoU >= 0.5 needs a prediction no more than twice the true length, so a 15s
# floor can never match a 5s event, of which the practice set has nine.
#
# Replaced by the only assumption we actually need: a symmetric buffer, below.


def extent_buffer(cfg=None) -> float:
    """Half-width of the uncertainty around a detected boundary, in seconds.

    This is the ONE declared constant left in the extent path, and it is a
    quantisation allowance rather than a guess about duration: frames arrive on
    a 1/sample_fps grid, so a true boundary can sit up to one sampling interval
    outside the window that caught it, and the window itself is built from
    whichever frames the sampler happened to land on.

    Default 2.0s. Note the tension - a larger buffer helps a boundary we
    straddled and hurts a very short event, because IoU falls as the prediction
    outgrows the truth. On a 2.6s event (T032 has one) a 2s buffer already
    costs the match. It lives in CFG so it can be swept offline against stored
    window_verdicts without another GPU run.
    """
    cfg = cfg or CFG
    return float(getattr(cfg, "extent_buffer_sec", 2.0))


def _recovered_between(health_curve, t_a: float, t_b: float, loose: float) -> bool:
    """RETIRED as a splitting criterion. Kept for diagnostics only.

    The idea was sound and the signal is not. Measured on the five-video run,
    comparing health inside a real ground-truth event against health outside it
    in the same video:

        T026  inside +0.134  outside +0.608   separation +0.474   works
        T031  inside -0.085  outside -0.092   separation -0.007   flat
        T032  inside +0.531  outside +0.415   separation -0.116   INVERTED
        T025  inside +0.150  outside -0.097   separation -0.247   INVERTED

    On three of four videos the frames containing the anomaly are as healthy as
    or HEALTHIER than the rest of the video - event frames sit at the 65th
    percentile of their own video's health. A congested road looks like a road;
    a person loitering looks like a person. Only T026's road spill, a plainly
    visible appearance change, separates at all.

    Used as a split test this was catastrophic on T031: health_thresh is
    calibrated globally at about -0.4, so loose is about -0.16, and T031's
    *minimum* health over 656 samples is -0.119. Every sample reads as
    "recovered", so eighteen agreeing congestion windows became eighteen events
    and eighteen false alarms. The best contiguous subset of those same windows
    scores IoU 0.917.
    """
    if not health_curve:
        return False
    mids = [h for t, h in health_curve if t_a < t < t_b]
    if not mids:
        return False
    return sum(h >= loose for h in mids) > len(mids) / 2


def measure_extent(cluster: list[dict], health_curve, thresh: float,
                   duration_sec: float | None = None,
                   cfg=None) -> tuple[float, float, str]:
    """How long was it? Answer from the windows and the curve, nothing else.

    Two measurements, composed:
      1. the span of the windows that actually saw it - direct evidence, and
         previously thrown away in favour of a prior
      2. extended outward while the health curve stays depressed - the scene
         itself telling us where it returned to normal

    ...plus a symmetric quantisation buffer. No floor, no cap, no prior. A
    one-window event is as short as that window; a forty-window event is as long
    as they span; and if the curve says the depression continues past the last
    window, so does the event.

    Returns (start, end, source) with source "measured" when the curve was
    consulted and "windows" when there was no curve to consult.
    """
    buf = extent_buffer(cfg)
    # A window may carry its own span - the interval its evidence actually
    # covers. The probe judges PROBE_SPAN_SEC of video at a time, so when it is
    # what fired, claiming only the ~2s stage 2 looked at understates what we
    # know. Measured on T025: probe spans reach IoU 0.800 against five of its
    # six real 20s events; the 2s window buffered to 6s reaches 0.30 and fails
    # the gate however right the class is. Windows without a span fall back to
    # their own bounds, so nothing changes where the probe was not involved.
    lo = [w["span"][0] if w.get("span") else w["t0"] for w in cluster]
    hi = [w["span"][1] if w.get("span") else w["t1"] for w in cluster]
    s, e = min(lo), max(hi)
    src = "probe-span" if any(w.get("span") for w in cluster) else "windows"

    if health_curve:
        ts = [t for t, _ in health_curve]
        hs = [h for _, h in health_curve]
        loose = thresh * EXTENT_LOOSE_FACTOR
        a = next((j for j, t in enumerate(ts) if t >= s), len(ts) - 1)
        while a > 0 and hs[a - 1] < loose:
            a -= 1
        b = next((j for j in range(len(ts) - 1, -1, -1) if ts[j] <= e), 0)
        while b < len(hs) - 1 and hs[b + 1] < loose:
            b += 1
        s, e = min(s, ts[a]), max(e, ts[b])
        src = "measured"

    s, e = s - buf, e + buf
    if s < 0:
        s = 0.0
    if duration_sec is not None and e > duration_sec:
        e = float(duration_sec)
        s = min(s, max(0.0, e - 1e-3))
    return s, e, src


def cluster_windows(windows: list[dict], health_curve=None,
                    thresh: float | None = None) -> list[list[dict]]:
    """Group same-class detections into one incident. Split only on class.

    We do not split on time, because a gap constant cannot express the question
    (T031 needs windows at 9s and 209s separated but 249s and 348s joined). We
    no longer split on the health curve either, because that curve is flat or
    inverted on three of four measured videos - see _recovered_between().

    So there is currently NO validated signal for where one incident ends and
    the next begins, and this asserts none. Measured on the five-video run,
    which is the whole justification:

        rule                          matched   false alarms   events
        health recovery (shipped)        0           20          20
        per-video relative recovery      0           12          12
        merge same class  <- this        0            2           2
        any time gap >= 60s              0            2           2

    Nothing recovers a match, because only 1 of the 15 ground-truth events is
    reachable from these windows at all. But asserting eighteen boundaries we
    cannot support costs eighteen false alarms, and the arena charges for each
    one. When the evidence does not distinguish, claim less.

    This becomes wrong the moment a video genuinely contains two separate
    incidents of the same class - T025 has six - so it is a stopgap, and the
    thing that unblocks it is a health signal that actually tracks anomalies,
    i.e. the linear probe.

    health_curve and thresh are accepted and ignored, so the call sites and the
    diagnostics that pass them keep working.
    """
    ws = sorted(windows, key=lambda w: w["t0"])
    hits = [w for w in ws if w["class"] != "normal"]
    clusters, cur = [], []
    for w in hits:
        if cur:
            prev = cur[-1]
            # Split where STAGE 2 ITSELF said normal. That is a judgement from
            # the model that can see the scene, it needs no constant, and it is
            # the only recovery signal we have left now the health curve has
            # been shown to be flat or inverted. Measured on the 34-video run it
            # recovers T033's cross-class match (smoke at 507s and an
            # altercation at 523s with nothing normal between them are one
            # incident) at the same false-alarm count as splitting on class.
            if any(x["class"] == "normal" and prev["t1"] <= x["t0"] and x["t1"] <= w["t0"]
                   for x in ws):
                clusters.append(cur)
                cur = []
        cur.append(w)
    if cur:
        clusters.append(cur)
    return clusters


def aggregate_events(windows: list[dict], cfg=None, health_curve=None,
                     duration_sec: float | None = None,
                     adjudicator=None) -> list[dict]:
    """Turn per-window verdicts into incidents with one main tag and sub-tags.

    windows: [{"t0","t1","class","confidence","description"}, ...]
    health_curve: [(t, health), ...] for measuring extent; None -> fallback prior
    adjudicator: optional fn(cluster) -> {"primary","confidence","reason"} used
        when a cluster holds several classes. NO causal table is consulted -
        the same class is a symptom in one context and the incident itself in
        another (smoke over a wreck vs smoke over a thermal plant), so the
        judgement is asked of the model that can see the context.
    """
    cfg = cfg or CFG
    thresh = cfg.health_thresh if cfg.health_thresh is not None else -0.4
    out = []

    for cluster in cluster_windows(windows, health_curve, thresh):
        strong = [w for w in cluster if w["confidence"] >= cfg.enter_conf]
        if not strong:
            continue                       # hysteresis: nothing opened this cluster

        classes = {}
        for w in cluster:
            c = classes.setdefault(w["class"], {"n": 0, "conf": 0.0, "first_t": w["t0"],
                                                "desc": w.get("description", "")})
            c["n"] += 1
            c["conf"] = max(c["conf"], w["confidence"])

        best = max(strong, key=lambda w: w["confidence"])
        main, main_conf, reason = best["class"], best["confidence"], "highest confidence"

        if len(classes) > 1 and adjudicator is not None:
            verdict = adjudicator(cluster)
            if verdict:
                main = verdict["primary"]
                main_conf = max(main_conf, verdict["confidence"])
                reason = verdict.get("reason", "adjudicated")

        # The span of the agreeing windows is itself a measurement of duration and
        # is now where measure_extent STARTS, rather than something it has to
        # override afterwards. The old window-span override existed only because
        # the fallback prior kept discarding this evidence; with the prior gone
        # there is nothing to override.
        start, end, src = measure_extent(strong, health_curve, thresh,
                                         duration_sec, cfg)

        # TEMPORAL survives the de-hardcoding on purpose, and the distinction is
        # worth being explicit about: it never invents a duration, it only
        # SUPPRESSES a claim that is physically implausible for its class (a
        # 0.3s fire). Every value is <= 4s and below the shortest real event we
        # have, so it should never fire in practice - if it starts firing, that
        # is a signal worth reading, not a threshold worth raising.
        if (end - start) + 1e-6 < TEMPORAL.get(main, DEFAULT_MIN_DUR):
            continue                       # too brief to be this class

        sub_tags = [{"class_name": c, "peak_confidence": round(v["conf"], 3),
                     "n_windows": v["n"], "first_seen_sec": round(v["first_t"], 2),
                     "description": v["desc"]}
                    for c, v in sorted(classes.items(), key=lambda kv: -kv[1]["conf"])
                    if c != main]

        out.append({
            "class_name": main,
            "start_time_sec": round(start, 2),
            "end_time_sec": round(end, 2),
            "confidence": round(float(np.mean([w["confidence"] for w in strong])), 3),
            "peak_confidence": round(float(main_conf), 3),
            "n_windows": len(cluster),
            "extent_source": src,           # "measured" (curve consulted) | "windows"
            "primary_reason": reason,
            "sub_tags": sub_tags,           # kept for analysis, not for the arena JSON
            "description_summary": best.get("description", ""),
        })
    return sorted(out, key=lambda e: e["start_time_sec"])


def _selftest():
    mk = lambda c, t, conf: {"t0": t, "t1": t + 1.0, "class": c,
                             "confidence": conf, "description": ""}

    # hysteresis: weak windows alone open nothing
    assert not aggregate_events([mk("fire", t, 0.4) for t in range(5)]), \
        "sub-enter_conf windows must not open an event"

    buf = extent_buffer()

    # NO FLOOR. A lone 1s detection is a ~1s event plus the buffer either side -
    # not the 15s the old prior asserted. This is the change that makes the
    # nine sub-10s ground-truth events reachable at all.
    ev = aggregate_events([mk("traffic_accident", 5.0, 0.9)])[0]
    span = ev["end_time_sec"] - ev["start_time_sec"]
    assert abs(span - (1.0 + 2 * buf)) < 1e-6, f"expected measured span, got {span}"
    assert ev["extent_source"] == "windows", "no curve given -> say so"

    # NO CAP. A genuinely long event stays long: E027's truth is one event over
    # ~600s of a 602s video, which the old 180s cap made unmatchable.
    long_curve = [(t, -0.5) for t in range(0, 600, 2)]
    ev = aggregate_events([mk("traffic_congestion", t, 0.9) for t in range(10, 580, 20)],
                          health_curve=long_curve, duration_sec=602.0)[0]
    assert ev["end_time_sec"] - ev["start_time_sec"] > 500, \
        f"a 600s event must survive aggregation, got {ev}"

    # SAME CLASS NEVER SPLITS, whatever the curve does. This is the T031 shape,
    # and it asserts a deliberate retreat: an earlier version split here on a
    # health recovery, which turned 18 agreeing congestion windows into 18 false
    # alarms on the real video because that curve is flat (its whole range sits
    # above the "recovered" line). Until a signal exists that actually tracks
    # anomalies, we assert one incident rather than eighteen boundaries we
    # cannot support.
    recov = [(t, 0.2 if 120 <= t <= 220 else -0.5) for t in range(0, 320, 2)]
    same = [mk("traffic_congestion", t, 0.9) for t in (20, 60, 100, 240, 280, 300)]
    assert len(aggregate_events(same, health_curve=recov, duration_sec=320.0)) == 1, \
        "same class must not split on a curve we have shown to be unreliable"
    flat = [(t, -0.5) for t in range(0, 320, 2)]
    assert len(aggregate_events(same, health_curve=flat, duration_sec=320.0)) == 1, \
        "...and the same with no recovery at all"

    # A class change alone does NOT split: T033's smoke at 507s and altercation
    # at 523s are one incident seen twice, and the adjudicator exists to name it.
    mixed = [mk("smoke", 507.5, 0.95), mk("fighting_or_violence", 523.5, 0.90)]
    assert len(cluster_windows(mixed)) == 1, "a class change alone is not a boundary"

    # ...but stage 2 saying "normal" in between IS a boundary, and it is the only
    # recovery signal left. Measured: this recovers T033's match at no extra cost.
    with_normal = [mk("traffic_congestion", 20.0, 0.9),
                   mk("normal", 100.0, 0.9),
                   mk("traffic_congestion", 200.0, 0.9)]
    assert len(cluster_windows(with_normal)) == 2, \
        "a normal verdict between two detections separates them"

    # CROSS-CLASS, T033's real shape: smoke then an altercation 16s later with
    # health depressed across the interval is one incident seen twice. The class
    # change is not what decides it - the curve is.
    t033_curve = [(t, -0.5 if 500 <= t <= 530 else 0.2) for t in range(400, 600)]
    t033 = [mk("smoke", 507.5, 0.95), mk("fighting_or_violence", 523.5, 0.90)]
    evs = aggregate_events(t033, health_curve=t033_curve, duration_sec=628.8)
    assert len(evs) == 1, "one incident, not two"
    assert evs[0]["class_name"] == "smoke"
    assert [s["class_name"] for s in evs[0]["sub_tags"]] == ["fighting_or_violence"]

    # ...and with an adjudicator the model's call wins, sub-tags still kept
    evs = aggregate_events(t033, health_curve=t033_curve, duration_sec=628.8,
                           adjudicator=lambda c: {"primary": "traffic_accident",
                                                  "confidence": 0.8,
                                                  "reason": "aftermath"})
    assert evs[0]["class_name"] == "traffic_accident"
    assert {s["class_name"] for s in evs[0]["sub_tags"]} == {"smoke", "fighting_or_violence"}

    # the curve extends an event beyond the window that caught it
    curve = [(t, -0.5 if 490 <= t <= 535 else 0.2) for t in range(400, 600)]
    ev = aggregate_events([mk("traffic_accident", 510.0, 0.9)],
                          health_curve=curve, duration_sec=628.8)[0]
    assert ev["extent_source"] == "measured"
    assert ev["start_time_sec"] <= 495 and ev["end_time_sec"] >= 530, \
        f"measured extent should track the low-health region, got {ev}"

    # a short event stays short - the whole point of removing the floor
    short_curve = [(t / 2, -0.5 if 30 <= t / 2 <= 35 else 0.2) for t in range(0, 200)]
    ev = aggregate_events([mk("traffic_accident", 32.0, 0.9)],
                          health_curve=short_curve, duration_sec=237.0)[0]
    assert ev["end_time_sec"] - ev["start_time_sec"] < 12.0, \
        f"a 5s event must not be inflated past IoU range, got {ev}"

    # expansion must never leave the video
    late = aggregate_events([mk("traffic_accident", 99.0, 0.9)], duration_sec=100.0)[0]
    assert late["end_time_sec"] <= 100.0 and late["start_time_sec"] >= 0.0

    print("temporal aggregation self-test passed (10 cases)")


_selftest()


## 8 — End to end

`process_video()` runs stage 0 → 1 → 2 → aggregation and reports a realtime
factor per video.

`LIMIT = 3` deliberately. Prove the wiring on three videos before spending
GPU-hours; raise it to `None` for the full 34-video public test set.

In [ ]:
# =============================================================================
# 7 - End to end: one video in, events out
# =============================================================================

def group_escalations(kept: list[dict], thresh: float, cfg=None) -> list[list[dict]]:
    """Bundle contiguous low-health frames into windows for stage 2.

    Per-frame VLM calls would be wasteful and would also throw away the temporal
    evidence the model needs - "is this vehicle stationary" is unanswerable from
    one frame. A window of a few frames spanning a couple of seconds answers it.
    """
    cfg = cfg or CFG
    gap = 2.0 / max(cfg.sample_fps, 0.1)      # allow one dropped sample inside a window
    flagged = [k for k in kept if k["health"] < thresh]
    windows, cur = [], []
    for k in flagged:
        if cur and (k["t"] - cur[-1]["t"]) > gap:
            windows.append(cur)
            cur = []
        cur.append(k)
    if cur:
        windows.append(cur)
    return windows


def add_scan_floor(windows: list[list[dict]], kept: list[dict],
                   duration: float, cfg=None) -> list[tuple[list[dict], str]]:
    """Guarantee a long video is looked at every scan_floor_interval_sec.

    Returns [(frames, source), ...] where source is "escalated" or "scan", so a
    detection can be traced back to which mechanism found it - that tag is the
    whole point of the experiment: it lets us say what the floor actually cost
    and gained, rather than watching the score move and guessing why.

    The floor only ADDS windows. Escalation is untouched, so disabling this
    returns behaviour to exactly what it was.
    """
    cfg = cfg or CFG
    out = [(w, "escalated") for w in windows]
    if not kept:
        return out

    def _never_looked():
        """Never report a video normal without the VLM having seen it once.

        Measured: 9 videos produced zero windows - no escalation, and too short
        for the interval floor - and SEVEN of those nine were genuinely
        anomalous (T007 accident, T008/T009 congestion, T010 stalled vehicle,
        T022 fighting, T023/T024 loitering). That was 41% of all our misses
        coming from videos we simply never examined. Nine extra VLM calls, ~50
        seconds, is a trivial price for removing a whole failure mode.

        The frames offered are the LOWEST-HEALTH ones available, so the single
        look gets the most suspicious moment rather than an arbitrary one.
        """
        worst = sorted(kept, key=lambda k: k["health"])[:cfg.vlm_frames]
        return [(sorted(worst, key=lambda k: k["t"]), "last-resort")]

    if not (cfg.scan_floor_enabled and duration > cfg.scan_floor_min_video_sec):
        return out or _never_looked()

    covered = [(w[0]["t"], w[-1]["t"]) for w in windows]
    step = cfg.scan_floor_interval_sec
    times = np.array([k["t"] for k in kept])

    t = 0.0
    while t < duration:
        slot_end = t + step
        # skip a slot an escalated window already covers - no point paying twice
        if any(a < slot_end and b >= t for a, b in covered):
            t = slot_end
            continue
        centre = t + step / 2
        # the frames nearest this slot's centre, in time order
        idx = np.argsort(np.abs(times - centre))[:cfg.vlm_frames]
        picked = [kept[i] for i in sorted(idx.tolist())]
        if picked and abs(picked[0]["t"] - centre) <= step:   # slot has real frames
            out.append((picked, "scan"))
        t = slot_end
    return out or _never_looked()


def pick_frames(window: list[dict], n: int) -> list[dict]:
    """Evenly spaced across the window, so the VLM sees change rather than n
    near-duplicates from the same instant."""
    if len(window) <= n:
        return window
    idx = np.linspace(0, len(window) - 1, n).round().astype(int)
    return [window[i] for i in sorted(set(idx.tolist()))]


def widen_window(kept: list[dict], centre_t: float, cfg=None) -> list[dict]:
    """cfg.vlm_frames frames spread over cfg.vlm_span_sec, centred on centre_t.

    An escalation window is ~2s wide because that is how often we sample, not
    because incidents last 2s. Handing the VLM only those frames is why it
    describes aftermath: a collision is over in about a second, and every later
    frame shows a queue. Widening to the probe's 16s gives both stages the same
    width of evidence and covers the median real event.

    Falls back to the window's own frames when the video is too short to widen,
    so a 6s L1 clip behaves exactly as before.
    """
    cfg = cfg or CFG
    span = float(getattr(cfg, "vlm_span_sec", 0.0) or 0.0)
    n = cfg.vlm_frames
    if span <= 0 or not kept:
        return []
    near = [k for k in kept if abs(k["t"] - centre_t) <= span / 2]
    if len(near) < 2:
        return []
    if len(near) <= n:
        return near
    idx = np.linspace(0, len(near) - 1, n).round().astype(int)
    return [near[i] for i in sorted(set(idx.tolist()))]


def _runtime_stats_since(model_name: str, start_idx: int) -> dict | None:
    """Slice CALL_LOG since this video started, for the arena's model_runtimes.

    Snapshotting the starting index rather than clearing CALL_LOG keeps a full
    run-long history intact (useful for our own diagnostics) while still giving
    an accurate per-video breakdown - the two uses don't conflict.
    """
    times = CALL_LOG.get(model_name, [])[start_idx:]
    if not times:
        return None
    arr = np.array(times)
    return {
        "model_name": model_name,
        "call_count": len(arr),
        "total_time_ms": round(float(arr.sum()), 1),
        "average_time_ms": round(float(arr.mean()), 1),
        "p50_time_ms": round(float(np.percentile(arr, 50)), 1),
        "p95_time_ms": round(float(np.percentile(arr, 95)), 1),
        "max_time_ms": round(float(arr.max()), 1),
    }


def process_video(path, cfg=None, verbose=False) -> dict:
    cfg = cfg or CFG
    _log_start = {k: len(v) for k, v in CALL_LOG.items()}
    t_start = time.time()
    kept, n_seen = stage1_video(path, cfg)
    t_stage1 = time.time() - t_start

    thresh = cfg.health_thresh
    if thresh is None and kept:
        # No calibration available: fall back to a within-video percentile. Worse
        # than calibrating on known-normal footage, because a video that is
        # anomalous throughout still escalates only escalate_pct of itself.
        thresh = float(np.percentile([k["health"] for k in kept], cfg.escalate_pct))

    windows = group_escalations(kept, thresh, cfg) if kept else []
    duration_full = video_duration(path)
    to_look_at = add_scan_floor(windows, kept, duration_full, cfg)

    # One capture reused across every window of this video - reopening the file
    # per frame would cost more than the seeks themselves.
    _native_cap = (cv2.VideoCapture(str(path))
                   if getattr(cfg, "vlm_crop_to_motion", False) else None)

    results = []
    t_vlm0 = time.time()
    for w, source in to_look_at:
        centre_t = (w[0]["t"] + w[-1]["t"]) / 2
        # Widen to cfg.vlm_span_sec where the video allows it, else fall back to
        # the escalation window itself. `widened` is truthy only when the VLM
        # actually judged the wider span, which is what makes the event extent
        # below an honest claim rather than an assumed one.
        widened = widen_window(kept, centre_t, cfg)
        picked = widened or pick_frames(w, cfg.vlm_frames)
        # Crop to the motion region at NATIVE resolution where we can. The
        # stored frame is already downscaled to max_side, which on 1280x720
        # source throws away 75% of the pixels - and the evidence that separates
        # a collision from the queue behind it lives in those pixels. One seek
        # per frame stage 2 actually looks at, about 39s across a five-video
        # run, and the token count does not change because the crop is
        # downscaled only if it is still larger than max_side.
        pil = []
        for r in picked:
            crop = native_crop(_native_cap, r["t"], r["box"],
                               (r["frame"].shape[1], r["frame"].shape[0]), cfg)
            if crop is not None:
                pil.append(to_pil(crop))      # already centred on the motion
            else:
                pil.append(to_pil(draw_visual_prompt(r["frame"], r["box"],
                                                     cfg.visual_prompt)))
        emb = embed_images(pil)
        # The probe sees 16s around this moment, not the ~2s the VLM sees, and
        # it costs nothing: stage 1 already encoded these frames and cell 5 now
        # keeps the vectors. Scored BEFORE the VLM so its shortlist can steer
        # the question, and kept afterwards so it can contradict the answer.
        pw = probe_window(kept, centre_t, cfg) if "probe_window" in globals() else {}
        p_anom = float(1.0 - pw.get("normal", 1.0)) if pw else 0.0
        cands = shortlist_classes(emb)
        try:
            verdict = vlm_verify(pil, cands)
        except Exception as e:
            print(f"  ! vlm failed on window @{w[0]['t']:.1f}s: "
                  f"{str(e).splitlines()[0][:120]}")
            continue

        # --- the probe may overrule a "normal" verdict, and only that ---------
        # Stage 2 answered "normal" on 79% of the windows that overlapped a real
        # event, including all 28 windows on T025's six accidents and T032's
        # four loitering events - with every class on offer. The probe reaches
        # 100% held-out recall on loitering and 85% on congestion, so where it
        # is confident and stage 2 has abstained, silence is the worse answer.
        #
        # One direction only. The probe never overrides a POSITIVE call, because
        # stage 2 looked at pixels and the probe looked at a mean of embeddings,
        # and it never fires below probe_override_p - measured at 0.95 that is
        # 353 of 484 held-out anomalies caught with zero false positives on 75
        # held-out normal clips. Held-out training clips are not 240-second test
        # videos from other cameras, so this is deliberately stricter than the
        # escalation bar and switchable from CFG.
        overrode = False
        top3 = sorted(((c, v) for c, v in pw.items() if c != "normal"),
                      key=lambda kv: -kv[1])[:3] if pw else []
        if (getattr(cfg, "probe_override_enabled", False) and top3
                and verdict["class"] == "normal"
                and p_anom >= getattr(cfg, "probe_override_p", 0.95)):
            chosen, conf = top3[0][0], float(p_anom)
            # The probe decides WHETHER; the VLM decides WHICH. Measured on
            # T025: the probe localises five of six real accidents at IoU 0.800
            # and calls every one of them wrong_way_driving. Its top-1 is 0.739
            # against 0.884 for its top-3, so re-ask with those three and no
            # "normal" - the question it failed at was never "is anything
            # happening", it was "which of these is it".
            # Widen the HINTS as well: the probe top-3 was 91% confident and wrong
            # on T025, so give the VLM the probe ranking plus stage 1's own,
            # deduplicated. The schema already offers all eleven either way.
            hints = list(dict.fromkeys([c for c, _ in top3] + list(cands)))[:6]
            pick = vlm_pick_class(pil, hints)
            if pick:
                chosen = pick["class"]
                conf = max(0.5, min(float(pick.get("confidence", conf)), conf))
            verdict = {**verdict, "class": chosen, "anomaly": True,
                       "confidence": round(conf, 3),
                       "description": (pick or {}).get("description")
                       or verdict.get("description", "")
                       or f"probe: {chosen.replace('_', ' ')}"}
            overrode = True

        results.append({
            "t0": w[0]["t"],
            "t1": w[-1]["t"] + 1.0 / cfg.sample_fps,
            "class": verdict["class"],
            "confidence": verdict["confidence"],
            "description": verdict["description"],
            "candidates": cands,
            "source": source,          # "escalated" | "scan" - the experiment's
                                       # whole point: traceable back to mechanism
            "health": float(np.mean([r["health"] for r in picked])),
            # both signals stored side by side, so the next question - which of
            # these two was right, and where - is answerable offline
            "probe_anomaly": round(p_anom, 4),
            "probe_top": top3[0][0] if top3 else None,
            "probe_top3": [[c, round(v, 4)] for c, v in top3],
            "probe_override": overrode,
            # What interval does the evidence actually cover? Stage 2 looked at
            # ~2s; the probe looked at PROBE_SPAN_SEC. When the probe is what
            # fired, the honest claim is its span, and that is worth a great
            # deal: probe-span extents score IoU 0.800 against five of T025's
            # six real 20s events, where a 2s window buffered to 6s scores 0.30
            # and fails the gate no matter how right the class is.
            # Set whenever the VLM actually judged the wider span - not only on
            # an override. If the model looked at 16s to reach its verdict, 16s
            # is the interval that verdict covers, whichever stage said it.
            "span": ([round(picked[0]["t"], 2),
                      round(picked[-1]["t"] + 1.0 / cfg.sample_fps, 2)]
                     if widened else None),
        })
    if _native_cap is not None:
        _native_cap.release()
    t_vlm = time.time() - t_vlm0
    n_scan = sum(1 for _, s in to_look_at if s == "scan")

    # The CONTAINER duration, not "wherever the last surviving frame landed".
    # Those differ by up to 2.7s on the practice pack, always short, because the
    # motion gate can drop the tail of a video - measured against the arena's own
    # manifest: T025 237.6 vs 240.0, T033 626.1 vs 628.8. It mattered little while
    # a 180s cap kept every event away from the end; with that cap gone an event
    # can legitimately run to the final frame, and clamping it to a duration 2.7s
    # short trims real overlap off exactly the long D3 events that pay 5 marks
    # each. It is also what cell 10 falls back to when no manifest file is
    # present, which on the practice pack is always.
    duration = duration_full if duration_full else (
        kept[-1]["t"] + 1.0 / cfg.sample_fps if kept else 0.0)

    # The health curve is what lets aggregation MEASURE an event's extent rather
    # than assume it - it is computed per frame in stage 1 and was previously
    # thrown away after the escalate/skip decision.
    curve = [(k["t"], k["health"]) for k in kept] if kept else None

    def _adjudicate(cluster):
        """One extra VLM call to name the primary incident when a cluster holds
        several classes. Deliberately not a causal lookup table: smoke is a
        symptom over a wrecked car and the incident itself over a thermal plant,
        so the call belongs to the model that can see which one this is."""
        frames = []
        for w in cluster:
            near = [k for k in kept if w["t0"] <= k["t"] <= w["t1"]]
            if near:
                r = near[len(near) // 2]
                frames.append(to_pil(draw_visual_prompt(r["frame"], r["box"],
                                                        cfg.visual_prompt)))
        return adjudicate_primary(frames[:cfg.vlm_frames], cluster) if frames else None

    events = aggregate_events(results, cfg, health_curve=curve,
                              duration_sec=video_duration(path),
                              adjudicator=_adjudicate)
    wall = time.time() - t_start

    # Arena schema's per-video runtime block - required on every video, and the
    # only source of the latency bonus. end_to_end_internal_time_ms starts here,
    # after models are already loaded, matching the rule to exclude load/download
    # time. chunks_processed has no exact spec meaning for our design; mapped to
    # "how many discrete windows needed the heavier model", floored at 1 for a
    # video that never escalated but still had a full stage-0/1 pass.
    model_runtimes = [s for s in (
        _runtime_stats_since("siglip2-encoder", _log_start.get("siglip2-encoder", 0)),
        _runtime_stats_since("vision-language-model",
                             _log_start.get("vision-language-model", 0)),
    ) if s is not None]

    out = {
        "video_id": Path(path).stem,
        "duration_sec": round(duration, 2),
        "frames_sampled": n_seen,
        "frames_kept": len(kept),
        "windows_escalated": len(windows),
        "windows_scan_floor": n_scan,
        # raw per-window verdicts, kept so aggregation can be re-tuned offline in
        # seconds instead of an 8-minute GPU re-run per experiment
        "window_verdicts": results,
        # ...and the curve those verdicts were measured against. Without it an
        # offline replay can reproduce the CLUSTERING but not the EXTENT, since
        # both now consult the curve - which made the last round of sweeps
        # unable to test the thing they were sweeping. ~600 floats per video,
        # rounded to keep the JSON readable.
        "health_curve": [[round(t, 2), round(h, 4)] for t, h in (curve or [])],
        "escalation_rate": round(len(windows) and sum(len(w) for w in windows)
                                 / max(len(kept), 1) or 0.0, 4),
        "events": events,
        "is_anomaly": int(bool(events)),
        "class_name": (max(events, key=lambda e: e["peak_confidence"])["class_name"]
                       if events else "normal"),
        "sec_stage1": round(t_stage1, 2),
        "sec_stage2": round(t_vlm, 2),
        "sec_total": round(wall, 2),
        "realtime_factor": round(duration / wall, 2) if wall > 0 else 0.0,
        "runtime_metadata": {
            "frames_processed": n_seen,
            "chunks_processed": max(1, len(windows)),
            "end_to_end_internal_time_ms": round(wall * 1000, 1),
            "model_runtimes": model_runtimes,
        },
    }
    if verbose:
        print(f"{out['video_id']:16s} {duration:6.1f}s  kept {len(kept):4d}/{n_seen:4d}  "
              f"esc {len(windows):3d} scan {n_scan:3d}  -> {out['class_name']:32s} "
              f"{out['realtime_factor']:5.2f}x realtime")
    return out


# --- run over the public test set --------------------------------------------
# 34 videos / ~56 min, with ground truth published, so this is the only honest
# read on whether any of the above works before the private evaluation.
def run_split(gt: pd.DataFrame, limit: int | None = None, cfg=None,
              only: list[str] | None = None) -> pd.DataFrame:
    cfg = cfg or CFG
    vids = gt.drop_duplicates("video_id")[["video_id", "path"]].dropna(subset=["path"])
    if only:
        want = list(dict.fromkeys(only))
        vids = vids[vids.video_id.isin(want)]
        missing = [v for v in want if v not in set(vids.video_id)]
        if missing:
            print(f"! requested but not found: {missing}")
        print(f"subset: {len(vids)} of {len(want)} requested videos")
    if limit:
        vids = vids.head(limit)
    rows, t0 = [], time.time()
    for i, (_, r) in enumerate(vids.iterrows(), 1):
        print(f"[{i}/{len(vids)}] ", end="")
        try:
            rows.append(process_video(r["path"], cfg, verbose=True))
        except Exception as e:
            print(f"FAILED {r['video_id']}: {str(e).splitlines()[0][:140]}")
            rows.append({"video_id": r["video_id"], "is_anomaly": 0,
                         "class_name": "normal", "events": [], "error": str(e)[:200]})
    df = pd.DataFrame(rows)
    total_video = df.get("duration_sec", pd.Series(dtype=float)).sum()
    print(f"\n{len(df)} videos, {total_video / 60:.1f} min of footage "
          f"in {(time.time() - t0) / 60:.1f} min wall "
          f"({total_video / max(time.time() - t0, 1e-6):.2f}x realtime)")
    # --- what did the scan floor actually buy? -------------------------------
    # The number this experiment turns on. Without it we would see the score move
    # and be guessing which mechanism moved it.
    if "window_verdicts" in df:
        esc = scan = esc_hit = scan_hit = 0
        for _, r in df.iterrows():
            for v in (r.get("window_verdicts") or []):
                is_scan = v.get("source") == "scan"
                scan += is_scan
                esc += not is_scan
                if v["class"] != "normal":
                    scan_hit += is_scan
                    esc_hit += not is_scan
        print(f"  windows: {esc} escalated ({esc_hit} non-normal), "
              f"{scan} scan-floor ({scan_hit} non-normal)")
        if "windows_scan_floor" in df:
            covered = int((df["windows_scan_floor"] > 0).sum())
            print(f"  scan floor active on {covered} video(s)")

    if "sec_stage1" in df:
        print(f"  stage 1: {df['sec_stage1'].sum():.0f}s    "
              f"stage 2: {df['sec_stage2'].sum():.0f}s    "
              f"({100 * df['sec_stage2'].sum() / max(df['sec_total'].sum(), 1e-6):.0f}% "
              "of wall time in the VLM)")
    return df


# --- the iteration set -------------------------------------------------------
# Five hand-picked videos, ~23 min of footage, ~8 min of GPU. A full 34-video
# run takes 21 minutes, which is too slow to think with; these five were chosen
# to make each Tier-1 change either visibly work or visibly fail.
#
#   T025  D2  238s  6x traffic_accident @20s      12 windows, ALL said "normal"
#                   -> the answer-menu bug, six independent chances to see it lift
#   T032  D3  308s  4x loitering (2.6-37.6s)      16 windows, ALL said "normal"
#                   -> the class we have never once scored, and D3 pays 5 marks
#   T031  D3  360s  1x traffic_congestion 235-360 18 windows, congestion FOUND
#                   -> the one event aggregation can win: oracle IoU 0.812, we
#                      scored 0 by emitting 9-311s. Tests the de-hardcoding.
#   T026  D2  238s  4x mixed classes              CONTROL - we already match one
#                   -> regression detector, and four different classes at once
#   T030  D2  239s  NORMAL                        CONTROL - false alarms
#                   -> guards the 100% D2 precision that item 1 puts at risk
#
# Baseline over these five: 1 of 15 ground-truth events matched (T026's spill).
# Set to None for the full set once a change looks right.
ONLY_VIDEOS = ["T025", "T026", "T030", "T031", "T032"]

LIMIT = None
PRED = run_split(GT_TEST if not GT_TEST.empty else GT_TRAIN,
                 limit=LIMIT, only=ONLY_VIDEOS)
_pred_out = run_path("predictions_raw.json")
PRED.to_json(_pred_out, orient="records", indent=1)
print(f"\nwrote {_pred_out}")


## 9 — Score against the local public test set

Diagnostics only, against the T00x videos we can actually see ground truth
for. The **false-alarm rate gets its own line** rather than being buried in
accuracy — a model that wins on F1 by flagging everything has failed the
actual brief. Level 2/3 temporal scoring now uses the arena's real gate
(**IoU ≥ 0.5**, correct class, at most one predicted event may match — extra
overlapping fragments count *against* you), not a loose diagnostic threshold.

In [ ]:
# =============================================================================
# 9 - Score against the public ground truth
# =============================================================================
# The real arena submission is a different file entirely (JSON, private E00x
# video set, IoU>=0.5 gate) - see cell 10. This cell is purely local diagnostics
# against the public T00x test set, which is the only ground truth we can see.
# Reported separately by level, because they are different tasks:
#   level 1  is this video anomalous, and which class      (no timestamps)
#   level 2  ...plus when it happened                      (temporal IoU)
#   level 3  ...plus a description
#
# The false-alarm rate on normal videos is printed on its own line and not
# buried inside accuracy. The PS is blunt about it - "an alerting system that
# fires regularly on ordinary activity stops being used" - so a model that wins
# on F1 by flagging everything has failed the actual brief.

def evaluate(pred: pd.DataFrame, gt: pd.DataFrame) -> dict:
    if pred.empty or gt.empty:
        print("nothing to evaluate")
        return {}

    g = gt[gt["video_id"].isin(pred["video_id"])].copy()
    truth = (g.groupby("video_id")
              .agg(is_anomaly=("is_anomaly", "max"),
                   classes=("class_name", lambda s: sorted(set(s.dropna()) - {"normal"})))
              .reset_index())
    m = pred.merge(truth, on="video_id", suffixes=("_pred", "_true"))
    if m.empty:
        print("predictions and ground truth share no video_id")
        return {}

    yp = m["is_anomaly_pred"].astype(int).to_numpy()
    yt = m["is_anomaly_true"].astype(int).to_numpy()
    tp = int(((yp == 1) & (yt == 1)).sum())
    fp = int(((yp == 1) & (yt == 0)).sum())
    fn = int(((yp == 0) & (yt == 1)).sum())
    tn = int(((yp == 0) & (yt == 0)).sum())
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    n_normal = int((yt == 0).sum())
    far = fp / max(n_normal, 1)

    # class correct only counts where an anomaly was correctly detected at all
    hit = m[(yp == 1) & (yt == 1)]
    cls_ok = int(sum(r["class_name"] in r["classes"] for _, r in hit.iterrows()))
    cls_acc = cls_ok / max(len(hit), 1)

    print("=" * 66)
    print(f"LEVEL 1   videos={len(m)}   TP={tp} FP={fp} FN={fn} TN={tn}")
    print(f"  precision {prec:.3f}   recall {rec:.3f}   F1 {f1:.3f}")
    print(f"  class accuracy on correctly-detected anomalies: "
          f"{cls_acc:.3f}  ({cls_ok}/{len(hit)})")
    print(f"  FALSE ALARM RATE on normal videos: {far:.3f}  ({fp}/{n_normal})")

    # --- level 2/3: temporal ---------------------------------------------------
    # The arena's actual gate (submission PDF): an event counts ONLY when the
    # class is right AND IoU >= 0.5 - "if your interval sits inside the real
    # event it must cover at least half of it; if it swallows the real event it
    # must be no more than twice as long." At most one predicted event can match
    # a given ground-truth event; every other overlapping prediction for the
    # SAME event counts AGAINST you, not neutrally. And predicting anything at
    # all on a video that's truly normal at level 2/3 scores that video ZERO -
    # there is no partial credit for a false alarm there. Both are much harsher
    # than the plain precision/recall above, so they're broken out separately.
    timed_ids = set(g.dropna(subset=["start_time_sec", "end_time_sec"]).video_id)
    normal_ids = set(g[g["class_name"] == "normal"].video_id) - timed_ids
    l23_normal_but_flagged = 0
    for vid in normal_ids:
        p = pred[pred["video_id"] == vid]
        if not p.empty and (p.iloc[0].get("events") or []):
            l23_normal_but_flagged += 1
    if normal_ids:
        print(f"\nLEVEL 2/3 FALSE-ALARM CHECK (real videos, not level-1 pooled)")
        print(f"  normal videos where we predicted anything (scores that video "
              f"ZERO under the real rule): {l23_normal_but_flagged}/{len(normal_ids)}")

    timed = g.dropna(subset=["start_time_sec", "end_time_sec"])
    ious_loose, ious_strict, extra_fragments = [], [], 0
    if not timed.empty:
        for _, row in timed.iterrows():
            p = pred[pred["video_id"] == row["video_id"]]
            events = (p.iloc[0].get("events") or []) if not p.empty else []
            same_class = [e for e in events if e["class_name"] == row["class_name"]]

            def iou(e):
                inter = max(0.0, min(e["end_time_sec"], row["end_time_sec"])
                            - max(e["start_time_sec"], row["start_time_sec"]))
                union = (max(e["end_time_sec"], row["end_time_sec"])
                         - min(e["start_time_sec"], row["start_time_sec"]))
                return inter / max(union, 1e-6)

            scores = [iou(e) for e in same_class]
            best = max(scores) if scores else 0.0
            ious_loose.append(best)
            ious_strict.append(best >= 0.5)
            # every same-class predicted event beyond the single best match is a
            # fragment that scores against this ground-truth event, per the rule
            extra_fragments += max(0, len(scores) - 1)

        print(f"\nLEVEL 2/3 TEMPORAL   {len(ious_loose)} timed ground-truth events")
        print(f"  real gate, IoU>=0.5 AND correct class: "
              f"{sum(ious_strict)}/{len(ious_loose)} matched")
        print(f"  mean IoU among correct-class predictions (loose diagnostic, "
              f"not the real gate): {np.mean(ious_loose):.3f}")
        print(f"  extra same-class fragments beyond the best match "
              f"(count AGAINST you): {extra_fragments}")
    else:
        print("\nLEVEL 2/3 TEMPORAL   no timed ground truth in this split")

    # --- per class -----------------------------------------------------------
    print("\nper-class detection (ground truth -> predicted):")
    for cls in ANOMALY_CLASSES:
        sub = m[m["classes"].apply(lambda cs: cls in cs)]
        if sub.empty:
            continue
        got = int(sum(r["class_name"] == cls for _, r in sub.iterrows()))
        det = int(sub["is_anomaly_pred"].sum())
        print(f"  {cls:34s} n={len(sub):3d}  detected {det:3d}  correct class {got:3d}")

    return {"precision": prec, "recall": rec, "f1": f1, "class_acc": cls_acc,
            "false_alarm_rate": far,
            "mean_iou_loose": float(np.mean(ious_loose)) if ious_loose else None,
            "level23_strict_matches": int(sum(ious_strict)) if ious_loose else None,
            "level23_timed_events": len(ious_loose) if ious_loose else None,
            "level23_extra_fragments": extra_fragments if ious_loose else None,
            "level23_normal_but_flagged": l23_normal_but_flagged if normal_ids else None,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}


# Scoring needs labels. The eval pack ships none - deliberately - so there is
# nothing here to compute and a fabricated zero would be worse than a skip: it
# would look like a real measurement in the metrics file.
if not HAS_TRUTH:
    METRICS = None
    print("MODE='eval' - no ground truth in this pack, so there is nothing to")
    print("score. Recall/F1/IoU are unavailable BY DESIGN, not by failure.")
    print()
    print("What to check instead, before submitting:")
    print("  - cell 8's per-video table: does every video get a plausible verdict?")
    print("  - cell 8's window counts: is anything getting zero looks?")
    print("  - cell 10's validator: schema, levels and timestamps")
    print("  - cell 11: eyeball a few frames and one full video replay")
else:
    METRICS = evaluate(PRED, GT_TEST)
    _m = run_path("metrics.json")
    _m.write_text(json.dumps(METRICS, indent=2))
    print(f"wrote {_m}")


# The arena submission is a different, stricter schema (JSON, private video
# set, null-vs-numeric timestamps by level) - built and validated in cell 10.
print("\n(arena submission file is built in cell 10, not here - "
      "different schema, different video set)")


## 10 — The arena submission file

A different, stricter schema than anything above: JSON, not CSV; scored
against a **private** video set (`E001, E002, …`) from `manifest.json`,
downloaded from the arena's Benchmark tab — not the local `T00x` test set.
Drop the fetched manifest into `/kaggle/working/manifest.json`; until then this
cell uses the local test set's own ground truth as a stand-in so it's testable
now.

Two silent-rejection traps this builds around: a normal video is `"events": []`
— never `"class_name": "normal"` — and Level-1 timestamps must be `null`, not
omitted. And two scoring rules the *aggregation* step (cell 7) already exists
to satisfy: a false alarm on a truly-normal Level-2/3 video scores that video
**zero**, and fragmenting one real event into several predictions only lets the
best-overlapping one match — the rest count against you.

In [ ]:
# =============================================================================
# 10 - Arena submission file
# =============================================================================
# The real evaluation is NOT the CSV cell 9 used to look at itself - it's a JSON
# file matching the arena's exact schema. The practice pack (checked against the
# live Benchmark page) turns out to BE our local T00x test set - 34 videos,
# L1 24 / L2 6 / L3 4 - so this is genuinely submittable today, not blocked on
# an unseen video set. A later "final" round may swap in unseen videos (the
# general PDF's own example uses E001, E002, ...), so nothing here hard-codes
# the T0xx naming. Drop a fetched manifest.json into WORK (see MANIFEST_PATH)
# to use the arena's exact video/level list instead of the local stand-in.
#
# Scoring weight, from the live page: D1=25, D2=35, D3=40 of 100. 75 of 100
# points sit on the 10 timed videos, not the 24 untimed ones - see
# docs/SUBMISSION_ARENA.md for the full rules this cell builds around.
#
# Two rules that get a file silently rejected, not just scored low:
#   - a normal video is "events": [] - NEVER {"class_name": "normal"}
#   - Level-1 events carry start_time_sec/end_time_sec = null, not omitted and
#     not 0 - Level 2/3 require real numbers, end strictly greater than start
#
# And two scoring rules worth designing the AGGREGATION around, not just
# complying with here:
#   - predicting ANYTHING on a truly-normal Level-2/3 video scores that video
#     ZERO - there is no partial credit for a false alarm there
#   - several overlapping fragments for one real event only let the BEST one
#     match; the rest count AGAINST you - our per-class merge_gap in cell 7 is
#     exactly what keeps this from happening, so don't loosen it casually

# The arena's practice manifest, embedded: {video_id: (level, duration_sec)}.
#
# It is here rather than read from a file because the file lives in the repo and
# the notebook runs on Kaggle, where nothing mounts it - so practice mode was
# silently falling through to the ground-truth CSV, which carries levels but NOT
# durations. That left the end_time_sec bounds check using our own decoded
# duration, which is short by up to 2.7s (T033 626.1 against the arena's 628.8),
# and short durations trim real overlap off exactly the long D3 events that pay
# five marks each.
#
# 34 entries is small enough to embed and large enough that a divergence would
# matter, so _check_embedded_manifest() below cross-checks it against GT_TEST at
# import time rather than trusting it to stay correct.
EMBEDDED_MANIFEST = {
    "T001": (1, 5.7), "T002": (1, 5.8), "T003": (1, 14.0),
    "T004": (1, 14.0), "T005": (1, 5.7), "T006": (1, 17.0),
    "T007": (1, 22.7), "T008": (1, 5.7), "T009": (1, 5.8),
    "T010": (1, 13.2), "T011": (1, 11.2), "T012": (1, 5.8),
    "T013": (1, 5.8), "T014": (1, 5.8), "T015": (1, 5.8),
    "T016": (1, 5.7), "T017": (1, 5.7), "T018": (1, 11.1),
    "T019": (1, 13.3), "T020": (1, 26.1), "T021": (1, 20.3),
    "T022": (1, 16.0), "T023": (1, 20.0), "T024": (1, 16.0),
    "T025": (2, 240.0), "T026": (2, 240.0), "T027": (2, 240.0),
    "T028": (2, 240.0), "T029": (2, 240.0), "T030": (2, 240.0),
    "T031": (3, 360.0), "T032": (3, 307.7), "T033": (3, 628.8),
    "T034": (3, 376.5),
}

# In eval mode the manifest ships inside the dataset itself, so there is nothing
# to fetch by hand and no chance of scoring against a stale copy. A file dropped
# into WORK still wins, which is the escape hatch if the arena reissues one.
_WORK_MANIFEST = WORK / "manifest.json"
MANIFEST_PATH = (_WORK_MANIFEST if _WORK_MANIFEST.exists()
                 else (EVAL_MANIFEST_PATH if MODE == "eval" and EVAL_MANIFEST_PATH
                       else _WORK_MANIFEST))


def _manifest_videos() -> tuple[list[dict], str]:
    """[{video_id, level, duration_sec}, ...] plus where it came from.

    One resolver for both loaders below, so the level map and the duration map
    can never disagree about which source they read.
    """
    if MANIFEST_PATH.exists():
        m = json.loads(MANIFEST_PATH.read_text())
        videos = m.get("videos", m if isinstance(m, list) else [])
        # the general PDF calls this field "level"; the live benchmark page's
        # own prose calls it "difficulty" - accept either rather than guess
        return ([{"video_id": v["video_id"],
                  "level": int(v.get("level", v.get("difficulty"))),
                  "duration_sec": (float(v["duration_sec"])
                                   if "duration_sec" in v else None)}
                 for v in videos], str(MANIFEST_PATH))
    if MODE != "eval" and EMBEDDED_MANIFEST:
        return ([{"video_id": k, "level": lv, "duration_sec": d}
                 for k, (lv, d) in sorted(EMBEDDED_MANIFEST.items())],
                "embedded practice manifest")
    return [], "none"


_MANIFEST_VIDEOS, MANIFEST_SOURCE = _manifest_videos()
print(f"manifest: {MANIFEST_SOURCE}  ({len(_MANIFEST_VIDEOS)} videos)")


def _check_embedded_manifest() -> None:
    """Cross-check the embedded copy against the mounted ground truth.

    An embedded constant is a copy, and copies drift. This is cheap and turns a
    silent wrong-level submission into a printed warning.
    """
    if MANIFEST_SOURCE != "embedded practice manifest" or GT_TEST.empty:
        return
    gt_lv = (GT_TEST.drop_duplicates("video_id")
             .set_index("video_id")["level"].astype(int).to_dict())
    emb = {v["video_id"]: v["level"] for v in _MANIFEST_VIDEOS}
    if set(emb) != set(gt_lv):
        print(f"  ! embedded manifest / ground truth disagree on which videos "
              f"exist: only-embedded={sorted(set(emb) - set(gt_lv))}, "
              f"only-truth={sorted(set(gt_lv) - set(emb))}")
    bad = {k: (emb[k], gt_lv[k]) for k in set(emb) & set(gt_lv) if emb[k] != gt_lv[k]}
    if bad:
        print(f"  ! embedded manifest / ground truth disagree on levels: {bad}")
    if not bad and set(emb) == set(gt_lv):
        print(f"  embedded manifest agrees with ground truth on all "
              f"{len(emb)} videos (ids and levels)")


_check_embedded_manifest()


def load_manifest() -> dict[str, int]:
    """{video_id: level} for every video the arena wants an answer for."""
    if _MANIFEST_VIDEOS:
        return {v["video_id"]: v["level"] for v in _MANIFEST_VIDEOS}
    print("no manifest available - using the local test set's ground truth as a "
          "stand-in so this cell is testable right now")
    return (GT_TEST.drop_duplicates("video_id")
            .set_index("video_id")["level"].astype(int).to_dict())


def load_manifest_durations() -> dict[str, float]:
    """video_id -> duration_sec, straight from the manifest when we have one -
    more authoritative than our own decoded duration for the 'end_time_sec must
    stay inside the duration' check. Falls back to PRED's measured duration."""
    return {v["video_id"]: v["duration_sec"] for v in _MANIFEST_VIDEOS
            if v.get("duration_sec") is not None}


def events_for_submission(pred_row: dict, level: int) -> list[dict]:
    """Our internal event dict -> the arena's exact per-event schema.

    Level 1 gets ONE event, never more - "One label for the whole clip" is the
    spec, not a suggestion. A second guess on a single-label task has zero
    possible upside (Level 1 scoring gives no credit for extra classes) and
    real downside (the arena counts each non-matching predicted event as its
    own false alarm, on top of the miss it doesn't fix) - measured directly: a
    tied-confidence fire+smoke double-guess on one real practice-pack video
    was two of the six false alarms in our first submission, when emitting
    only the correct one of the two would have cost nothing. Collapsing to the
    single highest-confidence event can only reduce that count, never raise it.
    """
    events = pred_row.get("events") or []
    if level == 1 and len(events) > 1:
        events = [max(events, key=lambda e: e["peak_confidence"])]
    out = []
    for e in events:
        # sub_tags stay OUT of the arena event object - the schema wants one
        # class per event - but they are real observations, so they go into the
        # explanation, which is a scored bonus field that never costs anything.
        expl = e.get("description_summary") or ""
        subs = [s["class_name"] for s in (e.get("sub_tags") or [])]
        if subs:
            also = ", ".join(s.replace("_", " ") for s in subs)
            expl = (expl + f" Also observed at this incident: {also}.").strip()
        expl = expl[:500] if len(expl) >= 20 else (expl or None)

        out.append({
            "class_name": e["class_name"],          # never "normal" - empty list instead
            "start_time_sec": None if level == 1 else float(e["start_time_sec"]),
            "end_time_sec": None if level == 1 else float(e["end_time_sec"]),
            "explanation": expl,
        })
    return out


def build_submission(pred: pd.DataFrame, manifest: dict[str, int],
                     submission_id: str, model_name: str = "ahc-cascade-v1") -> dict:
    by_id = {r["video_id"]: r for r in pred.to_dict("records")}
    predictions, total_wall_ms, max_parallel = [], 0.0, 1

    for vid, level in manifest.items():
        row = by_id.get(vid)
        if row is None:
            print(f"  ! {vid}: not in PRED - omitted. Per the rules an omitted "
                  "video KEEPS its previous answer (or scores normal if you have "
                  "never answered it) - it is not cleared.")
            continue
        events = events_for_submission(row, level)
        rt = row.get("runtime_metadata") or {
            "frames_processed": row.get("frames_sampled", 0),
            "chunks_processed": 1,
            "end_to_end_internal_time_ms": round(row.get("sec_total", 0) * 1000, 1),
            "model_runtimes": [],
        }
        total_wall_ms += rt["end_to_end_internal_time_ms"]
        predictions.append({"video_id": vid, "events": events, "runtime_metadata": rt})

    return {
        "schema_version": "1.0",
        "submission_id": submission_id,
        "model_name": model_name,
        "run_metadata": {"total_wall_time_ms": round(total_wall_ms, 1),
                         "hardware": GPU_NAME, "max_parallel_videos": max_parallel},
        "predictions": predictions,
    }


def validate_submission(sub: dict, manifest: dict[str, int],
                        durations: dict[str, float] | None = None) -> list[str]:
    """Every rule from the PDF's 'Things that catch people out', checked before
    upload. A rejected file doesn't burn a run, but there's no reason to find
    that out on the arena instead of here.

    `durations` (video_id -> seconds), when given, also checks the live
    benchmark page's rule that end_time_sec must "stay inside the duration" -
    a check the general PDF never mentions, so it's easy to miss.
    """
    durations = durations or {}
    problems, seen = [], set()
    for p in sub["predictions"]:
        vid = p["video_id"]
        if vid in seen:
            problems.append(f"{vid}: video_id appears more than once")
        seen.add(vid)
        if vid not in manifest:
            problems.append(f"{vid}: not in manifest")
            continue
        level = manifest[vid]
        if "runtime_metadata" not in p:
            problems.append(f"{vid}: missing runtime_metadata (required on "
                            "every video; also where the latency bonus comes from)")
        dur = durations.get(vid)
        for e in p["events"]:
            if e["class_name"] not in ANOMALY_CLASSES:
                problems.append(f"{vid}: class_name {e['class_name']!r} invalid - "
                                "must be one of the 11 event classes, never 'normal'")
            if level == 1:
                if e["start_time_sec"] is not None or e["end_time_sec"] is not None:
                    problems.append(f"{vid}: Level 1 events must have null timestamps")
            else:
                if e["start_time_sec"] is None or e["end_time_sec"] is None:
                    problems.append(f"{vid}: Level {level} requires real timestamps")
                elif e["end_time_sec"] <= e["start_time_sec"]:
                    problems.append(f"{vid}: end_time_sec must be greater than start_time_sec")
                elif dur is not None and e["end_time_sec"] > dur + 0.5:
                    problems.append(f"{vid}: end_time_sec {e['end_time_sec']} exceeds "
                                    f"the video's duration ({dur}s)")
    missing = set(manifest) - seen
    if missing:
        problems.append(f"{len(missing)} manifest video(s) never answered "
                        f"(scored as normal by default): {sorted(missing)[:10]}"
                        f"{' ...' if len(missing) > 10 else ''}")
    return problems


MANIFEST = load_manifest()
SUBMISSION = build_submission(PRED, MANIFEST, submission_id="ahc-run-01")
VIDEO_DURATIONS = (load_manifest_durations()
                  or dict(zip(PRED["video_id"], PRED.get("duration_sec", []))))
PROBLEMS = validate_submission(SUBMISSION, MANIFEST, VIDEO_DURATIONS)

OUT_PATH = run_path("arena_submission.json")
OUT_PATH.write_text(json.dumps(SUBMISSION, indent=1))
size_kb = OUT_PATH.stat().st_size / 1024
print(f"\nwrote {OUT_PATH}  ({size_kb:.1f} KB of the 5 MB cap, "
      f"{len(SUBMISSION['predictions'])} videos)")

if PROBLEMS:
    print(f"\n{len(PROBLEMS)} problem(s) - fix before uploading:")
    for p in PROBLEMS[:30]:
        print(f"  ! {p}")
else:
    print("no problems found by local validation - still spot-check a few "
          "entries by eye before uploading, this checks format, not judgment")


## 11 — See it, don't just read the JSON

A grid of real frames: one per detected event (predicted class + confidence,
green border if the class matches ground truth, red if it doesn't), plus a
few genuinely missed anomalies for honest contrast. Doubles as the example
frames the architecture write-up and 2-slide PPT are asked to include —
"prefer visuals over long text."

In [ ]:
# =============================================================================
# 11 - See it, don't just read the JSON
# =============================================================================
# Two controlled outputs, not a dump of everything:
#
#   GALLERY_VIDEO_IDS  - stills, one per video, shown individually. Edit this
#                        list to whichever videos you want to eyeball - only
#                        these get an image, nothing is auto-selected.
#
#   LIVE_CHECK_VIDEO_ID - ONE video gets replayed frame-by-frame with the
#                        motion gate, health score and any final alert overlaid
#                        over time, written out as a real mp4 and played inline.
#                        Re-samples and re-scores that one video (stage 0+1
#                        only - no extra VLM calls, the alert overlay reuses
#                        PRED's already-decided events). Costs seconds, not
#                        the several minutes a full re-run would.

import matplotlib.pyplot as plt
from IPython.display import Video, display


def grab_frame_at(path, t_sec):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, int(t_sec * fps)))
    ok, frame = cap.read()
    cap.release()
    return frame if ok else None


def put_text(frame, lines, origin=(10, 30), color=(0, 0, 255), scale=0.7):
    out = frame
    x, y = origin
    for line in lines:
        cv2.putText(out, line, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(out, line, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 1, cv2.LINE_AA)
        y += int(28 * scale / 0.7)
    return out


def gt_label(video_id: str) -> str:
    # "unknown", not "normal". On the eval pack every class_name is NA, and the
    # old code's `else "normal"` would have quietly asserted that all 28 videos
    # are normal - colouring every prediction red as a false positive and
    # inverting the meaning of the whole gallery.
    if not HAS_TRUTH:
        return "unknown (no truth in this pack)"
    rows = GT_TEST[GT_TEST.video_id == video_id]
    if rows.empty:
        return "no ground truth"
    cls = sorted(set(rows.class_name.dropna()) - {"normal"})
    return ", ".join(cls) if cls else "normal"


def pred_row(video_id: str) -> dict | None:
    hits = [r for r in PRED.to_dict("records") if r["video_id"] == video_id]
    return hits[0] if hits else None


# --- controlled still gallery --------------------------------------------
# Edit freely. Nothing outside this list gets rendered.
# Practice ids are T0xx and eval ids are E0xx, so a single hard-coded list
# renders an empty gallery in one of the two modes. Both are spelled out; edit
# whichever applies. In eval mode this deliberately picks one video per level
# plus the longest, because with cell 9 unable to score anything these frames
# are the only check on the run before it is submitted.
GALLERY_VIDEO_IDS = (["E003", "E017", "E021", "E025", "E028"] if MODE == "eval"
                     else ["T005", "T012", "T016", "T009", "T033"])
GALLERY_VIDEO_IDS = [v for v in GALLERY_VIDEO_IDS if v in VIDEO_PATHS]

for vid in GALLERY_VIDEO_IDS:
    path = VIDEO_PATHS.get(vid)
    if path is None:
        print(f"! {vid} not found, skipping"); continue
    row = pred_row(vid) or {}
    events = row.get("events") or []
    if events:
        ev = max(events, key=lambda e: e["peak_confidence"])
        t = (ev["start_time_sec"] + ev["end_time_sec"]) / 2
        pred_txt = f"pred: {ev['class_name']} ({ev['confidence']:.2f})"
        # blue = "we cannot say if this is right"; green/red only where truth exists
        color = ((0, 140, 200) if not HAS_TRUTH else
                 (0, 180, 0) if ev["class_name"] in gt_label(vid) else (0, 0, 255))
    else:
        gt_rows = GT_TEST[GT_TEST.video_id == vid]
        t = (gt_rows.iloc[0].start_time_sec if not gt_rows.empty
             and pd.notna(gt_rows.iloc[0].start_time_sec) else video_duration(path) / 2)
        # This one does not crash on an all-NA column - pandas returns an empty
        # frame - but "not missed" would then be an assertion we cannot support,
        # so the truthless case short-circuits rather than relying on that.
        missed = HAS_TRUTH and vid in set(
            GT_TEST[GT_TEST.is_anomaly == True].video_id)
        pred_txt = "pred: normal" + (" (missed)" if missed else "")
        color = (0, 140, 200) if not HAS_TRUTH else (
            (0, 140, 255) if missed else (0, 180, 0))

    frame = grab_frame_at(path, t)
    if frame is None:
        print(f"! {vid} frame grab failed, skipping"); continue
    frame = put_text(frame, [vid, pred_txt, f"truth: {gt_label(vid)}"], color=color)

    plt.figure(figsize=(6, 4.2))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(vid)
    plt.axis("off")
    out_path = RUNS / "gallery" / f"{vid}_{RUN_ID}.png"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=110, bbox_inches="tight")
    plt.show()
    print(f"wrote {out_path}")


# --- one video, played back with the pipeline's own reasoning overlaid ------
LIVE_CHECK_VIDEO_ID = "E028" if MODE == "eval" else "T033"


def make_live_check_video(video_id: str, cfg=None) -> Path | None:
    cfg = cfg or CFG
    path = VIDEO_PATHS.get(video_id)
    if path is None:
        print(f"! {video_id} not found"); return None

    row = pred_row(video_id) or {}
    events = row.get("events") or []
    thresh = cfg.health_thresh if cfg.health_thresh is not None else -0.4

    kept, n_seen = sample_video(path, cfg)
    if not kept:
        print(f"! nothing survived the motion gate for {video_id}"); return None
    imgs = [to_pil(draw_visual_prompt(k["frame"], k["box"], cfg.visual_prompt)) for k in kept]
    health_scores = health(embed_images(imgs)).tolist()

    h, w = kept[0]["frame"].shape[:2]
    out_path = run_path(f"live_check_{video_id}.mp4")
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"),
                             max(cfg.sample_fps, 2.0), (w, h))

    for k, hscore in zip(kept, health_scores):
        t = k["t"]
        frame = draw_visual_prompt(k["frame"], k["box"], cfg.visual_prompt).copy()
        escalated = hscore < thresh
        lines = [f"t={t:6.1f}s", f"health={hscore:+.3f}" + ("  ESCALATED" if escalated else "")]
        active = next((e for e in events if e["start_time_sec"] <= t <= e["end_time_sec"]), None)
        color = (0, 0, 255) if escalated else (0, 200, 0)
        if active:
            lines.append(f"ALERT: {active['class_name']} ({active['confidence']:.2f})")
            color = (0, 0, 255)
        frame = put_text(frame, lines, color=color)
        writer.write(frame)
    writer.release()

    print(f"{video_id}: {len(kept)} frames, {len(events)} final event(s), wrote {out_path}")
    return out_path


live_path = make_live_check_video(LIVE_CHECK_VIDEO_ID)
if live_path is not None:
    display(Video(str(live_path), embed=True, html_attributes="controls width=640"))


# --- incident timeline: the main tag AND what else was seen ------------------
# The arena JSON only carries one class per event. That is the right answer for
# scoring and the wrong answer for a person on the other end of an alert, who
# wants to know an accident happened AND that there is smoke and a crowd. The
# sub-tags exist for that reader; this is where they become visible.

def plot_incident_timeline(video_id: str):
    row = pred_row(video_id)
    if not row or not row.get("events"):
        print(f"{video_id}: no incidents to plot"); return
    dur = row.get("duration_sec") or video_duration(VIDEO_PATHS[video_id])
    events = row["events"]

    fig, ax = plt.subplots(figsize=(11, 1.1 + 0.75 * len(events)))
    ax.set_xlim(0, dur); ax.set_ylim(-0.5, len(events) - 0.5)
    ax.set_xlabel("seconds"); ax.set_yticks([])
    ax.set_title(f"{video_id} - incidents, with what else was observed", fontsize=11)

    for i, e in enumerate(events):
        s, en = e["start_time_sec"], e["end_time_sec"]
        measured = e.get("extent_source", "").startswith("measured")
        ax.barh(i, en - s, left=s, height=0.42,
                color="#c44" if measured else "#c88",
                hatch=None if measured else "//", edgecolor="#822")
        ax.text(s, i + 0.30, f"{e['class_name']}  ({e['confidence']:.2f})",
                fontsize=9, weight="bold", va="bottom")
        # extent provenance matters: a measured span is evidence, the fallback
        # is a prior, and the plot should not let those look the same
        ax.text(en + dur * 0.005, i, "measured" if measured else "fallback 10s",
                fontsize=7.5, va="center", color="#666")
        for j, sub in enumerate(e.get("sub_tags") or []):
            ax.plot([sub["first_seen_sec"]], [i - 0.22 - j * 0.1], marker="v",
                    ms=6, color="#48c")
            ax.text(sub["first_seen_sec"] + dur * 0.004, i - 0.24 - j * 0.1,
                    f"also: {sub['class_name']} ({sub['peak_confidence']:.2f})",
                    fontsize=7.5, va="center", color="#26a")

    gt_rows = GT_TEST[(GT_TEST.video_id == video_id) & GT_TEST.start_time_sec.notna()]
    for _, g in gt_rows.iterrows():
        ax.axvspan(g.start_time_sec, g.end_time_sec, color="#2a2", alpha=0.13, zorder=0)
    if len(gt_rows):
        ax.text(0.99, 1.06, "green band = ground truth", transform=ax.transAxes,
                ha="right", fontsize=8, color="#2a2")

    plt.tight_layout()
    out = RUNS / "gallery" / f"timeline_{video_id}_{RUN_ID}.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=110, bbox_inches="tight")
    plt.show()
    print(f"wrote {out}")

    print(f"\n{video_id} incident detail:")
    for e in events:
        print(f"  [{e['start_time_sec']:7.1f} - {e['end_time_sec']:7.1f}] "
              f"{e['class_name']:32s} conf {e['confidence']:.2f}  "
              f"({e.get('extent_source','?')})")
        if e.get("primary_reason"):
            print(f"      why this tag: {e['primary_reason']}")
        for sub in (e.get("sub_tags") or []):
            print(f"      also seen: {sub['class_name']:28s} "
                  f"conf {sub['peak_confidence']:.2f} @ {sub['first_seen_sec']:.1f}s")


for _vid in [v for v in GALLERY_VIDEO_IDS if (pred_row(v) or {}).get("events")]:
    plot_incident_timeline(_vid)

# full structure, sub-tags included, for later analysis - deliberately separate
# from the arena file, which only ever carries the single main tag per event
_inc = run_path("incidents_detailed.json")
_inc.write_text(json.dumps(
    [r for r in PRED.to_dict("records") if r.get("events")], indent=1, default=str))
print(f"\nwrote {_inc} (main tags + sub-tags + provenance)")
